# Dissecting Pre-MZT Cis-Regulatory Grammar via Cross-Species Sequence-to-Function Transfer Learning

[![Python 3.12](https://img.shields.io/badge/python-3.12-blue.svg)](https://www.python.org/downloads/release/python-3120/)
[![PyTorch 2.11](https://img.shields.io/badge/PyTorch-2.11%20CUDA-ee4c2c.svg)](https://pytorch.org/)
[![Conda](https://img.shields.io/badge/conda-nucleotide-green.svg)](https://docs.conda.io/)

---

### Abstract & Workflow Overview
During animal embryogenesis, developmental control transitions from maternal transcripts to nascent zygotic transcription in two distinct waves: the **minor wave (pre-MZT)** and the canonical **major wave (MZT)**. Decoding the cis-regulatory grammar and pioneer transcription factor (TF) motifs driving early accessibility before widespread chromatin opening has been hindered by low cell numbers, cytoplasmic polyadenylation, and rapid embryonic cell cycles.

This repository implements a publication-ready, species- and locus-agnostic bioinformatics framework to decode the cis-regulatory grammar of pre-MZT gene activation via cross-species transfer learning across teleosts (*Danio rerio*, *Oryzias latipes*, and *Astatotilapia calliptera*).

The analysis unfolds across 9 cohesive blocks:
- **Block 0: Study Configuration, Compute Profile & Task Manager** (strictly centralized settings)
- **Block 1: Target-Species RNA-seq Analysis** (Replicate-aware / median-of-ratios normalization, corrected 5'-to-3' spliced coverage diagnostic, 3' UTR CPE test & exonic SNP shift check)
- **Block 1b: Zebrafish SLAM-seq Nascent RNA Ortholog Overlay** (MA Plot & Polyadenylation / Early ZGA Diagnostic)
- **Block 2: Background ATAC-seq Preprocessing & ChromBPNet Training** (*Danio rerio* & *Oryzias latipes*)
- **Block 3: Stage 2 Continuous Pre-MZT Linear Probe & Grafted End-to-End Models** (All-peak vs repeat-filtered excluding miR-430)
- **Block 4: Zero-Shot Validation on Target-Species Minor-Wave Genes & Medaka ATAC** (Primary clean sets vs secondary diagnostic)
- **Block 5: Unbiased Sliding-Window Scan of `TARGET_LOCUS_NAME`**
- **Block 6: Base-Resolution Attribution, Teleost-Normalized Pre-MZT TF Filtering & Evolutionary Swap**
- **Block 7: Multi-Task `NTv3` Fine-Tuning, Baseline Benchmarks, QuantSeq 3'-End Scoring & Distal Knockout**

In [ ]:
# ==============================================================================
# Emergency Task Termination Cell (Self-Contained)
# ==============================================================================
# Set TERMINATE_ALL_TASKS = True and run this cell to terminate all running background tasks.
# Guarded with False by default so accidental 'Run All' never terminates active jobs.
TERMINATE_ALL_TASKS = False

import sys
from pathlib import Path
if "." not in sys.path:
    sys.path.insert(0, ".")

try:
    from scripts.task_manager import TaskManager
    cand_work_dirs = [p for p in Path(".").glob("work_*") if p.is_dir() and (p / "tasks").exists()]
    active_work_dir = str(cand_work_dirs[0]) if cand_work_dirs else "./work_astatotilapia_premzt"
    tm_emergency = TaskManager(work_dir=active_work_dir)
    if TERMINATE_ALL_TASKS:
        print("[!] TERMINATE_ALL_TASKS is True: Terminating all active background tasks...")
        tm_emergency.cancel_all()
    else:
        print("[*] Current TaskManager status (TERMINATE_ALL_TASKS=False):")
    display(tm_emergency.status())
except Exception as e:
    print(f"[-] TaskManager check notice: {e}")


---
## Block 0: Study Configuration, Compute Profile & Task Manager

All study-specific settings (target species, target locus, gene lists, BAM paths, reference resources, pre-MZT TF lists, and compute scaling) live strictly in this block. No study-specific logic or filenames are hard-coded in subsequent analysis blocks.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import json
import time
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import pysam

# ==============================================================================
# 1. Study & Target Locus Settings
# ==============================================================================
STUDY_NAME = "astatotilapia_premzt"
TARGET_SPECIES_NAME = "Astatotilapia_calliptera"
TARGET_LOCUS_NAME = "notch2_slc35a3a"
TARGET_GENES = ["notch2", "slc35a3a"]
LOCUS_FLANK_BP = 75000
CANONICAL_ZGA_MARKERS = ["vox", "klf17", "mxtx2"]

# Target RNA-seq BAMs (relative paths inside pwd)
RNASEQ_EARLY_BAMS = ["./RNA-seq/AC_2C_rep1.markdup.sorted.bam"]      # 2-cell maternal baseline
RNASEQ_PREMZT_BAMS = ["./RNA-seq/AC_1K_rep1.markdup.sorted.bam"]    # 1000-cell pre-MZT stage

# Target genome and GTF (set to local relative path, or None to auto-download)
TARGET_GENOME_FA = "./work_astatotilapia_premzt/downloads/Astatotilapia_calliptera.fAstCal1.3.dna_sm.toplevel.fa"
TARGET_GTF = "./work_astatotilapia_premzt/downloads/Astatotilapia_calliptera.fAstCal1.3.112.gtf.gz"

# Pre-MZT TF expression filter
PREMZT_TF_LIST_PATH = "./TF_gene_names.txt"

# SLAM-seq & Ortholog Mapping options
SLAMSEQ_GENE_TABLE_PATH = None  # None -> auto-generated at WORK_DIR/slamseq/zf_slamseq_gene_classes.tsv
ORTHOLOG_MAP_PATH = None        # None -> auto-generated at WORK_DIR/downloads/orthologs_target_to_zf.tsv

# ==============================================================================
# 2. Epigenomic & Transcriptomic Dataset Accessions
# ==============================================================================
# Zebrafish ATAC-seq (GSE130944, Palfy et al. 2020)
ZEBRAFISH_ATAC_ACCESSIONS = [
    "GSM3756599", "GSM3756600",  # 256-cell early blastula replicates
    "GSM3756603", "GSM3756604", "GSM3756605",  # dome late blastula replicates
    "SRR9032640"  # naked genomic DNA Tn5 control
]

# Medaka ATAC-seq (PRJDB7492, Nakamura et al. 2021)
MEDAKA_ATAC_ACCESSIONS = [
    "DRX143405", "DRX143406", "DRX143407",  # st9 early blastula replicates
    "DRX143412", "DRX143413"               # st11 late blastula replicates
]

# Zebrafish Nascent SLAM-seq (PRJNA768450, Bhat et al. 2023)
SLAM_ACCESSIONS = [
    "GSM5610039", "GSM5610040",  # 0.75 hpf (+4sU) baseline
    "GSM5610043", "GSM5610044",  # 2.5 hpf (+4sU) pre-MZT
    "GSM5610045", "GSM5610046", "GSM5610047", "GSM5610048",  # 3.0-3.5 hpf (+4sU) onset
    "GSM5610051", "GSM5610052",  # 4.5 hpf (+4sU) major wave
    "GSM5610075", "GSM5610076", "GSM5610077"   # -4sU untreated controls
]

USE_PRETRAINED_BIAS = False
BIAS_MODEL_PATH = None

# ==============================================================================
# 3. Environment, Credentials & Compute Profile
# ==============================================================================
CONDA_ENV_NAME = "nucleotide"
COMPUTE_PROFILE = "laptop_16gb"  # Options: "laptop_16gb" (default) or "hpc_1mb"

# Hugging Face token (redacted from logs/saved JSON; sets HF_HOME inside WORK_DIR)
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Compute & execution parameters
WORK_DIR = f"./work_{STUDY_NAME}"
THREADS = 8
FORCE_RERUN = False
CV_FOLDS = 5
ENABLE_MEDAKA = True
EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2 = False
NTV3_RUN_BASELINES = True
ENABLE_OPTION2_TARGET_RNA_ABLATION = False

# Epigenomics & RNA-seq Quantification Settings
CHROMBPNET_EPOCHS = 15
RNASEQ_STRANDNESS = 0  # 0: unstranded, 1: stranded, 2: reversely stranded

# Profile presets
if COMPUTE_PROFILE == "laptop_16gb":
    NTV3_MODEL_ID = "InstaDeepAI/NTv3_tracks"
    NTV3_CONTEXT_LEN = 65536
    NTV3_TRAIN_MODE = "lora"
    NTV3_PRECISION = "bf16"
    NTV3_GRAD_CHECKPOINTING = True
    NTV3_BATCH_SIZE = 1
    NTV3_GRAD_ACCUM = 16
    NUM_GPUS = 1
    EXECUTION_BACKEND = "local_background"
    slurm_config = None
elif COMPUTE_PROFILE == "hpc_1mb":
    NTV3_MODEL_ID = "InstaDeepAI/NTv3_tracks_1mb"
    NTV3_CONTEXT_LEN = 1048576
    NTV3_TRAIN_MODE = "full"
    NTV3_PRECISION = "bf16"
    NTV3_GRAD_CHECKPOINTING = True
    NTV3_BATCH_SIZE = 4
    NTV3_GRAD_ACCUM = 4
    NUM_GPUS = 4
    EXECUTION_BACKEND = "slurm"
    slurm_config = {
        "partition": "gpu",
        "gres": f"gpu:{NUM_GPUS}",
        "cpus_per_task": THREADS,
        "mem": "64G",
        "time": "24:00:00"
    }
else:
    raise ValueError(f"Unknown COMPUTE_PROFILE: {COMPUTE_PROFILE}")

# Ensure HF_HOME is strictly inside WORK_DIR
os.environ["HF_HOME"] = os.path.abspath(f"{WORK_DIR}/hf_cache")

# ==============================================================================
# 4. Reference Resources (Auto-downloaded if not present)
# ==============================================================================
ZEBRAFISH_GENOME_FA = f"{WORK_DIR}/downloads/Danio_rerio.GRCz11.dna_sm.primary_assembly.fa"
ZEBRAFISH_GTF = f"{WORK_DIR}/downloads/Danio_rerio.GRCz11.112.gtf.gz"
MEDAKA_GENOME_FA = f"{WORK_DIR}/downloads/Oryzias_latipes.ASM223467v1.dna_sm.toplevel.fa"
MEDAKA_GTF = f"{WORK_DIR}/downloads/Oryzias_latipes.ASM223467v1.112.gtf.gz"
JASPAR_MEME_PATH = f"{WORK_DIR}/downloads/JASPAR2024_CORE_vertebrates_non-redundant.meme"

# Create directory hierarchy strictly inside pwd
for d in ["tasks", "downloads", "atac", "slamseq", "rnaseq", "models", "results", "hf_cache"]:
    Path(f"{WORK_DIR}/{d}").mkdir(parents=True, exist_ok=True)

# Define and create canonical results directory for target locus to prevent NameError
res_dir = Path(WORK_DIR) / "results" / TARGET_LOCUS_NAME
res_dir.mkdir(parents=True, exist_ok=True)

# Redact token before serializing config snapshot
config_snapshot = {
    "STUDY_NAME": STUDY_NAME,
    "TARGET_SPECIES_NAME": TARGET_SPECIES_NAME,
    "TARGET_LOCUS_NAME": TARGET_LOCUS_NAME,
    "TARGET_GENES": TARGET_GENES,
    "TARGET_GENOME_FA": TARGET_GENOME_FA,
    "TARGET_GTF": TARGET_GTF,
    "ZEBRAFISH_GENOME_FA": ZEBRAFISH_GENOME_FA,
    "ZEBRAFISH_GTF": ZEBRAFISH_GTF,
    "MEDAKA_GENOME_FA": MEDAKA_GENOME_FA,
    "MEDAKA_GTF": MEDAKA_GTF,
    "JASPAR_MEME_PATH": JASPAR_MEME_PATH,
    "PREMZT_TF_LIST_PATH": PREMZT_TF_LIST_PATH,
    "SLAMSEQ_GENE_TABLE_PATH": SLAMSEQ_GENE_TABLE_PATH,
    "ORTHOLOG_MAP_PATH": ORTHOLOG_MAP_PATH,
    "ZEBRAFISH_ATAC_ACCESSIONS": ZEBRAFISH_ATAC_ACCESSIONS,
    "MEDAKA_ATAC_ACCESSIONS": MEDAKA_ATAC_ACCESSIONS,
    "SLAM_ACCESSIONS": SLAM_ACCESSIONS,
    "USE_PRETRAINED_BIAS": USE_PRETRAINED_BIAS,
    "BIAS_MODEL_PATH": BIAS_MODEL_PATH,
    "COMPUTE_PROFILE": COMPUTE_PROFILE,
    "NTV3_MODEL_ID": NTV3_MODEL_ID,
    "NTV3_CONTEXT_LEN": NTV3_CONTEXT_LEN,
    "EXECUTION_BACKEND": EXECUTION_BACKEND,
    "CHROMBPNET_EPOCHS": CHROMBPNET_EPOCHS,
    "RNASEQ_STRANDNESS": RNASEQ_STRANDNESS,
    "CV_FOLDS": CV_FOLDS,
    "ENABLE_MEDAKA": ENABLE_MEDAKA,
    "NTV3_RUN_BASELINES": NTV3_RUN_BASELINES,
    "ENABLE_OPTION2_TARGET_RNA_ABLATION": ENABLE_OPTION2_TARGET_RNA_ABLATION,
    "HF_TOKEN": "***REDACTED***" if HF_TOKEN else ""
}
with open(f"{WORK_DIR}/config.json", "w") as f:
    json.dump(config_snapshot, f, indent=2)

# Initialize TaskManager (passing slurm_config when HPC profile selected)
from scripts.task_manager import TaskManager
tm = TaskManager(work_dir=WORK_DIR, backend=EXECUTION_BACKEND, slurm_config=slurm_config)

print(f"[+] Study '{STUDY_NAME}' configured successfully.")
print(f"    Target Species:  {TARGET_SPECIES_NAME}")
print(f"    Target Locus:    {TARGET_LOCUS_NAME} ({TARGET_GENES})")
print(f"    Results Dir:     {res_dir}")
print(f"    Compute Profile: {COMPUTE_PROFILE} (Backend: {EXECUTION_BACKEND})")
print(f"    HF Cache:        {os.environ['HF_HOME']}")
print(f"    ChromBPNet Ep:   {CHROMBPNET_EPOCHS} | RNA-seq Strandness: {RNASEQ_STRANDNESS}")
display(tm.status())

# ==============================================================================
# Target Genome Assembly & BAM @SQ Contig Length Consistency Check
# ==============================================================================
if Path(TARGET_GENOME_FA).exists():
    fai_path = Path(f"{TARGET_GENOME_FA}.fai")
    if not fai_path.exists():
        run_cmd = f"samtools faidx '{TARGET_GENOME_FA}'"
        subprocess.run(run_cmd, shell=True, check=True)
    fai_lens = {}
    with open(fai_path) as f:
        for line in f:
            p = line.strip().split("\t")
            if len(p) >= 2:
                fai_lens[p[0]] = int(p[1])

    all_target_bams = [b for b in RNASEQ_EARLY_BAMS + RNASEQ_PREMZT_BAMS if Path(b).exists()]
    for bam_p in all_target_bams:
        with pysam.AlignmentFile(bam_p, "rb") as bf:
            sq_headers = bf.header.get("SQ", [])
            for sq in sq_headers:
                cname = sq.get("SN")
                clen = sq.get("LN")
                if cname in fai_lens:
                    assert clen == fai_lens[cname], (
                        f"[FATAL] Assembly mismatch! BAM {Path(bam_p).name} contig '{cname}' length {clen} "
                        f"does not match TARGET_GENOME_FA contig length {fai_lens[cname]}!"
                    )
    print(f"[+] Verified BAM @SQ contig lengths strictly match {Path(TARGET_GENOME_FA).name}.")

In [ ]:
# ==============================================================================
# Consolidated Resource Download, Audit & Background Queue Block
# ==============================================================================
# Audits required study resources based strictly on Block 0 settings input.
# - Fast pre-download integrity checks: skips download if valid file already exists.
# - If downloading in background: re-running this cell prints current progress once.
# - If downloads are pending: submits background task (max 3 distinct host websites in parallel).
# - Includes lightweight header/chromosome consistency check across FASTA, GTF, and BAMs.
from scripts.download_resources import check_resource_status, download_all_resources

# Reference paths for downstream analysis blocks
filtered_meme_path = f"{WORK_DIR}/downloads/premzt_tfs.meme"
ortholog_tsv = f"{WORK_DIR}/downloads/orthologs_target_to_zf.tsv"
df_orthologs = pd.read_csv(ortholog_tsv, sep="\t") if Path(ortholog_tsv).exists() else None

# 1. Check if download_resources is ALREADY RUNNING FIRST
task_info = tm.get_task_info("download_resources")
task_status = task_info.get("status")

if task_status == "RUNNING":
    elapsed = task_info.get("elapsed_sec", 0)
    prog = task_info.get("progress", "-")
    print(f"[*] Task 'download_resources' is currently RUNNING in background (Elapsed: {elapsed:.1f}s)")
    print(f"    Current Progress: {prog}")
    prog_file = Path(WORK_DIR) / "tasks" / "download_resources" / "progress.txt"
    if prog_file.exists():
        try:
            with open(prog_file) as pf:
                print(f"    Progress details: {pf.read().strip()}")
        except Exception:
            pass
    print("\n--- TaskManager Live Tail (last 15 lines) ---")
    tm.tail("download_resources", n=15)
    print("\n--- TaskManager Status Table ---")
    display(tm.status())
    print("\n[!] Skipping resource audit while download task is actively writing files.")
    print("    Tip: Re-run this cell to refresh live progress, or execute tm.wait('download_resources') to wait for completion.")
else:
    # 2. Lightweight header/chromosome consistency check
    print("[*] Performing lightweight reference and BAM consistency audit...")
    try:
        fa_p = Path(TARGET_GENOME_FA)
        fa_chroms = set()
        if fa_p.exists():
            fai = Path(str(fa_p) + ".fai")
            if not fai.exists():
                subprocess.run(["samtools", "faidx", str(fa_p)], check=False)
            if fai.exists():
                with open(fai) as f:
                    fa_chroms = set(line.split("\t")[0] for line in f)

        # Sample first ~5,000 non-comment lines of GTF
        gtf_p = Path(TARGET_GTF)
        gtf_chroms = set()
        if gtf_p.exists():
            import gzip
            open_fn = gzip.open if str(gtf_p).endswith(".gz") else open
            with open_fn(gtf_p, "rt", errors="replace") as gf:
                lines_checked = 0
                for line in gf:
                    if line.startswith("#"):
                        continue
                    parts = line.split("\t")
                    if len(parts) > 0:
                        gtf_chroms.add(parts[0])
                    lines_checked += 1
                    if lines_checked >= 5000:
                        break

        bam_chroms = set()
        if RNASEQ_EARLY_BAMS and Path(RNASEQ_EARLY_BAMS[0]).exists():
            with pysam.AlignmentFile(RNASEQ_EARLY_BAMS[0], "rb") as bf:
                bam_chroms = set(bf.references)

        # Genuine consistency checks
        if fa_chroms and gtf_chroms:
            fa_gtf_overlap = fa_chroms.intersection(gtf_chroms)
            if not fa_gtf_overlap:
                print(f"  [!] WARNING: Zero chromosome overlap between FASTA ({len(fa_chroms)} contigs) and GTF ({len(gtf_chroms)} sampled contigs)!")
            else:
                print(f"  [+] FASTA and GTF chromosome consistency verified ({len(fa_gtf_overlap)} shared contigs).")

        if fa_chroms and bam_chroms:
            overlap = fa_chroms.intersection(bam_chroms)
            if not overlap:
                print(f"  [!] WARNING: Zero overlap between FASTA headers ({len(fa_chroms)} contigs) and BAM references ({len(bam_chroms)} contigs)!")
            else:
                print(f"  [+] Reference chromosomes verified: {len(overlap)} overlapping contigs between FASTA and BAM.")
    except Exception as e:
        print(f"  [-] Notice during reference audit: {e}")

    # 3. Audit status of required resources
    manifest_df, all_ready, pending_tasks = check_resource_status(
        work_dir=Path(WORK_DIR),
        target_species=TARGET_SPECIES_NAME,
        target_fa=TARGET_GENOME_FA,
        target_gtf=TARGET_GTF,
        zf_fa=ZEBRAFISH_GENOME_FA,
        zf_gtf=ZEBRAFISH_GTF,
        enable_medaka=ENABLE_MEDAKA,
        medaka_fa=MEDAKA_GENOME_FA,
        medaka_gtf=MEDAKA_GTF,
        jaspar_meme_path=JASPAR_MEME_PATH,
        premzt_tf_list=PREMZT_TF_LIST_PATH,
        slamseq_gene_table=SLAMSEQ_GENE_TABLE_PATH,
        ortholog_map_path=ORTHOLOG_MAP_PATH,
        skip_slamseq=False,
        atac_accessions_zf=ZEBRAFISH_ATAC_ACCESSIONS,
        atac_accessions_medaka=MEDAKA_ATAC_ACCESSIONS,
        slam_accessions=SLAM_ACCESSIONS,
        use_pretrained_bias=USE_PRETRAINED_BIAS
    )

    if all_ready:
        print("[+] All required study resources are present and verified intact.")
        display(manifest_df)
        display(tm.status())
    else:
        dl_cmd = [
            "python", "-u", "scripts/download_resources.py",
            "--work-dir", WORK_DIR,
            "--config", f"{WORK_DIR}/config.json",
            "--max-host-workers", "3",
            "--max-per-host-workers", "2"
        ]
        if not ENABLE_MEDAKA:
            dl_cmd.append("--disable-medaka")
        if SLAMSEQ_GENE_TABLE_PATH:
            dl_cmd.extend(["--slamseq-table", SLAMSEQ_GENE_TABLE_PATH])
        
        print(f"[*] Identified {len(pending_tasks)} resource(s) requiring download:")
        display(manifest_df[manifest_df["Status"].str.startswith("PENDING")])
        print("[*] Submitting 'download_resources' to TaskManager (Max 3 distinct host websites in parallel)...")
        tm.submit("download_resources", dl_cmd)
        time.sleep(1.0)
        display(tm.status())
        print("    Tip: Re-run this cell at any time to inspect live progress, or execute tm.wait('download_resources') to block.")


---
## Block 1: Target-Species RNA-seq Analysis (Minor-Wave Calling & Diagnostics)

We quantify maternal baseline (`AC_2C`) vs. pre-MZT (`AC_1K`) reads across all annotated genes in `TARGET_GTF`:
1. **Replicate-Aware Normalization / Median-of-Ratios Fallback:**
   - When $\ge 2$ replicates per stage are provided, runs full `PyDESeq2` (Wald test $p_{\text{adj}} < 0.05$ and LFC shrinkage).
   - When 1 BAM per stage is provided, computes DESeq2-style median-of-ratios size factors (geometric mean across stages) rather than composition-biased raw CPM, printing an explicit notice that replicate $p_{\text{adj}}$ is unavailable.
2. **Strict Minor-Wave vs. Control Gene Classification:**
   - `strictly_zygotic_minor_wave`: `cpm_early < 0.5` ($\le 5$ counts), `cpm_premzt >= 2.0` ($\ge 20$ counts), $\log_2\text{FC} \ge 2.0$.
   - `all_upregulated`: $\log_2\text{FC} \ge 1.5$, `cpm_premzt >= 2.0` (diagnostic/unfiltered set, may contain cytoplasmic polyadenylation or early major-wave transcripts).
   - `control_non_activated`: `cpm_early >= 10.0`, $|\log_2\text{FC}| \le 0.5$ (expression- and GC-matched).
3. **Corrected 5'-to-3' Spliced Exonic Coverage Diagnostic:**
   - Projects read depth onto concatenated spliced exonic coordinates (excluding introns).
   - **Strictly reverses bin order ($100\% \rightarrow 0\%$) for minus-strand ($-$) genes so $0\%$ is always the true 5' TSS and $100\%$ is always the true 3' TES.**
   - Automated strand sanity assertion confirming identical 5'→3' biological polarity on $+$ and $-$ strand genes.
4. **Post-Transcriptional vs. Transcriptional Diagnostics:**
   - **3' UTR CPE Enrichment Test:** Tests canonical CPEs (`TTTTAT`, `TTTTAAT`), U-rich eCPEs, and `AATAAA`/`ATTAAA` hexamers in `all_upregulated` vs `control_non_activated` (Fisher's exact test). Inspects $+500\text{ bp}$ downstream of `TARGET_GENES` for read extension into downstream PAS motifs.
   - **Exonic SNP Allele-Frequency Shift Check (`pysam`):** Scans exonic positions with depth $\ge 20$ for polymorphic allele shifts between stages (testing for paternal allele emergence).

In [ ]:
import math
import gzip
import re
import pysam
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path

def is_bam_paired_end(bam_path: str) -> bool:
    """Auto-detect if BAM contains paired-end reads."""
    try:
        with pysam.AlignmentFile(bam_path, "rb") as bf:
            for read in bf:
                if not read.is_secondary and not read.is_supplementary:
                    return bool(read.is_paired)
    except Exception:
        pass
    return False

# 1. FeatureCounts Quantification
counts_file = Path(WORK_DIR) / "rnaseq" / "counts.txt"
counts_file.parent.mkdir(parents=True, exist_ok=True)
if not counts_file.exists() or FORCE_RERUN:
    all_bams = RNASEQ_EARLY_BAMS + RNASEQ_PREMZT_BAMS
    is_pe = is_bam_paired_end(all_bams[0]) if all_bams and Path(all_bams[0]).exists() else False
    cmd = ["featureCounts", "-T", str(THREADS), "-s", str(RNASEQ_STRANDNESS)]
    if is_pe:
        cmd.append("-p")
    cmd.extend(["-a", TARGET_GTF, "-o", str(counts_file)] + all_bams)
    print(f"[*] Running featureCounts on target BAMs (Paired-End: {is_pe})...")
    subprocess.run(cmd, check=True)

df_counts = pd.read_csv(counts_file, sep="\t", comment="#")
early_bams = [Path(b).name for b in RNASEQ_EARLY_BAMS]
premzt_bams = [Path(b).name for b in RNASEQ_PREMZT_BAMS]

# Match column headers
col_early = [c for c in df_counts.columns if any(b in c for b in early_bams)]
col_premzt = [c for c in df_counts.columns if any(b in c for b in premzt_bams)]

# 2. Normalization: PyDESeq2 when replicated, Median-of-Ratios size factor fallback for unreplicated BAMs
if len(col_early) >= 2 and len(col_premzt) >= 2:
    print("[*] Replicates available: using PyDESeq2 for replicate-aware differential expression...")
    from pydeseq2.dds import DeseqDataSet
    from pydeseq2.ds import DeseqStats

    counts_df = df_counts[["Geneid"] + col_early + col_premzt].set_index("Geneid").T
    clinical_df = pd.DataFrame({
        "sample": counts_df.index,
        "condition": ["early"] * len(col_early) + ["premzt"] * len(col_premzt)
    }).set_index("sample")

    dds = DeseqDataSet(counts=counts_df, metadata=clinical_df, design_factors="condition", refit_cooks=True)
    dds.deseq2()
    stat_res = DeseqStats(dds, contrast=["condition", "premzt", "early"])
    stat_res.summary()
    res_df = stat_res.results_df

    df_counts["baseMean"] = res_df["baseMean"].values
    df_counts["log2fc"] = res_df["log2FoldChange"].values
    df_counts["pval"] = res_df["pvalue"].values
    df_counts["padj"] = res_df["padj"].values

    raw_early = df_counts[col_early].sum(axis=1)
    raw_premzt = df_counts[col_premzt].sum(axis=1)
    tot_early = df_counts[col_early].values.sum()
    tot_premzt = df_counts[col_premzt].values.sum()
    df_counts["cpm_early"] = (raw_early / max(tot_early, 1)) * 1e6
    df_counts["cpm_premzt"] = (raw_premzt / max(tot_premzt, 1)) * 1e6
else:
    print("[*] NOTICE: Unreplicated single-BAM per stage detected (AC_2C_rep1 vs. AC_1K_rep1).")
    print("    Computing robust geometric median-of-ratios size factors across stages...")
    c_early = df_counts[col_early[0]].values.astype(float)
    c_premzt = df_counts[col_premzt[0]].values.astype(float)

    # Geometric mean pseudo-reference across positive genes
    geom_ref = np.exp(0.5 * (np.log(np.maximum(c_early, 1e-5)) + np.log(np.maximum(c_premzt, 1e-5))))
    pos_mask = (c_early > 0) & (c_premzt > 0)
    sf_early = float(np.median((c_early[pos_mask] / geom_ref[pos_mask])))
    sf_premzt = float(np.median((c_premzt[pos_mask] / geom_ref[pos_mask])))

    print(f"    Size Factors: 2-Cell Early = {sf_early:.4f} | 1000-Cell Pre-MZT = {sf_premzt:.4f}")

    tot_early = c_early.sum()
    tot_premzt = c_premzt.sum()
    df_counts["cpm_early"] = (c_early / max(tot_early, 1)) * 1e6
    df_counts["cpm_premzt"] = (c_premzt / max(tot_premzt, 1)) * 1e6
    norm_early = c_early / max(sf_early, 1e-5)
    norm_premzt = c_premzt / max(sf_premzt, 1e-5)
    df_counts["log2fc"] = np.log2((norm_premzt + 1.0) / (norm_early + 1.0))
    df_counts["padj"] = 1.0

# 3. Gene Symbol Annotation from GTF
gene_name_map = {}
with (gzip.open(TARGET_GTF, "rt") if str(TARGET_GTF).endswith(".gz") else open(TARGET_GTF, "rt")) as f:
    for line in f:
        if line.startswith("#"):
            continue
        p = line.strip().split("\t")
        if p[2] == "gene":
            gid, gname = None, None
            for item in p[8].split(";"):
                item = item.strip()
                if item.startswith("gene_id"):
                    m = re.search(r'gene_id\s+"([^"]+)"', item)
                    if m:
                        gid = m.group(1)
                elif item.startswith("gene_name"):
                    m = re.search(r'gene_name\s+"([^"]+)"', item)
                    if m:
                        gname = m.group(1)
            if gid:
                gene_name_map[gid] = gname or gid

df_counts["gene_name"] = df_counts["Geneid"].map(gene_name_map).fillna(df_counts["Geneid"])
df_counts["gene_id"] = df_counts["Geneid"]
if "Chr" in df_counts.columns:
    df_counts["chrom"] = df_counts["Chr"].apply(lambda x: str(x).split(";")[0])
if "Strand" in df_counts.columns:
    df_counts["strand"] = df_counts["Strand"].apply(lambda x: str(x).split(";")[0])

# Parse min start and max end across all exon coordinates so minus-strand genes never get start > end
if "Start" in df_counts.columns and "End" in df_counts.columns:
    all_starts, all_ends = [], []
    for s_val, e_val in zip(df_counts["Start"], df_counts["End"]):
        s_coords = [int(x) for x in str(s_val).split(";") if x.isdigit()]
        e_coords = [int(x) for x in str(e_val).split(";") if x.isdigit()]
        coords = s_coords + e_coords
        if coords:
            all_starts.append(min(coords))
            all_ends.append(max(coords))
        else:
            all_starts.append(0)
            all_ends.append(0)
    df_counts["start"] = all_starts
    df_counts["end"] = all_ends
elif "Start" in df_counts.columns:
    df_counts["start"] = df_counts["Start"].apply(lambda x: int(str(x).split(";")[0]))
    df_counts["end"] = df_counts["End"].apply(lambda x: int(str(x).split(";")[-1]))

df_counts["tss"] = np.where(df_counts["strand"] == "+", df_counts["start"], df_counts["end"])
df_counts["tes"] = np.where(df_counts["strand"] == "+", df_counts["end"], df_counts["start"])

# 4. Standardized Pre-MZT Minor-Wave Calling Categories
c_early_val = df_counts[col_early].mean(axis=1).values if len(col_early) >= 2 else df_counts[col_early[0]].values
c_premzt_val = df_counts[col_premzt].mean(axis=1).values if len(col_premzt) >= 2 else df_counts[col_premzt[0]].values
cpm_early_val = df_counts["cpm_early"].values
cpm_premzt_val = df_counts["cpm_premzt"].values
lfc_val = df_counts["log2fc"].values

# Category A: strictly_zygotic_minor_wave (clean true zygotic activation)
is_strictly_zygotic = (
    (cpm_early_val < 0.5) &
    (c_early_val <= 5) &
    (cpm_premzt_val >= 2.0) &
    (c_premzt_val >= 20) &
    (lfc_val >= 2.0)
)
if len(col_early) >= 2 and len(col_premzt) >= 2:
    is_strictly_zygotic = is_strictly_zygotic & (df_counts["padj"].values < 0.05)

# Category B: all_upregulated (true superset including strictly zygotic genes)
is_all_upregulated = (
    (cpm_premzt_val >= 1.0) &
    (c_premzt_val >= 10) &
    (lfc_val >= 1.5)
)
if len(col_early) >= 2 and len(col_premzt) >= 2:
    is_all_upregulated = is_all_upregulated & (df_counts["padj"].values < 0.05)

# Category C: control_non_activated (expression- and length-matched silent genes)
is_control = (
    (cpm_early_val >= 2.0) &
    (c_early_val >= 20) &
    (np.abs(lfc_val) <= 0.5) &
    (~is_all_upregulated) &
    (~is_strictly_zygotic)
)

# Maintain boolean indicator columns across dataframes and exported TSVs
df_counts["is_strictly_zygotic_minor_wave"] = is_strictly_zygotic
df_counts["is_all_upregulated"] = is_all_upregulated
df_counts["is_control_non_activated"] = is_control

categories = np.array(["unclassified"] * len(df_counts), dtype=object)
categories[is_control] = "control_non_activated"
categories[is_all_upregulated] = "all_upregulated"
# Primary label for mutual exclusivity in discrete categorization:
categories[is_strictly_zygotic] = "strictly_zygotic_minor_wave"
df_counts["category"] = categories

print("=" * 80)
print(f"[*] Target Species Pre-MZT Minor-Wave Gene Calling Summary:")
print(f"    Total genes quantified:                 {len(df_counts):,}")
print(f"    Strictly zygotic minor wave (Primary):   {is_strictly_zygotic.sum():,}")
print(f"    All upregulated genes (Diagnostic):     {is_all_upregulated.sum():,}")
print(f"    Control non-activated genes (Baseline): {is_control.sum():,}")
print("=" * 80)

# Check target genes
for tg in TARGET_GENES:
    sub = df_counts[df_counts["gene_name"].str.lower() == tg.lower()]
    if not sub.empty:
        r = sub.iloc[0]
        print(f"    Target Gene '{tg}': 2-Cell CPM={r['cpm_early']:.2f}, 1000-Cell CPM={r['cpm_premzt']:.2f}, log2FC={r['log2fc']:.2f} -> Category: {r['category']}")

# Export gene sets table
gene_sets_tsv = res_dir / f"{TARGET_LOCUS_NAME}_gene_sets.tsv"
df_counts.to_csv(gene_sets_tsv, sep="\t", index=False)
alt_gene_sets = Path(WORK_DIR) / "rnaseq" / "target_species_gene_sets.tsv"
df_counts.to_csv(alt_gene_sets, sep="\t", index=False)
print(f"[+] Gene sets exported to {gene_sets_tsv}")

# Block 1 Volcano / MA Plot
import matplotlib.pyplot as plt

fig, (ax_ma, ax_vol) = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: MA plot
mean_cpm = 0.5 * (df_counts["cpm_early"] + df_counts["cpm_premzt"])
ax_ma.scatter(np.log10(mean_cpm + 0.1), df_counts["log2fc"], color="#cccccc", alpha=0.3, s=8, label="Other Genes")
ax_ma.scatter(np.log10(mean_cpm[is_control] + 0.1), df_counts.loc[is_control, "log2fc"], color="#2b83ba", alpha=0.6, s=12, label="Control Non-Activated")
ax_ma.scatter(np.log10(mean_cpm[is_all_upregulated & ~is_strictly_zygotic] + 0.1), df_counts.loc[is_all_upregulated & ~is_strictly_zygotic, "log2fc"], color="#fdae61", alpha=0.7, s=16, label="All Upregulated (Maternal+Zygotic)")
ax_ma.scatter(np.log10(mean_cpm[is_strictly_zygotic] + 0.1), df_counts.loc[is_strictly_zygotic, "log2fc"], color="#d7191c", alpha=0.8, s=22, label="Strictly Zygotic Minor Wave")

# Callout target genes & canonical ZGA markers
for gname in TARGET_GENES + CANONICAL_ZGA_MARKERS:
    sub = df_counts[df_counts["gene_name"].str.lower() == gname.lower()]
    if not sub.empty:
        r = sub.iloc[0]
        mx = np.log10(0.5 * (r["cpm_early"] + r["cpm_premzt"]) + 0.1)
        my = r["log2fc"]
        ax_ma.scatter([mx], [my], color="black", s=60, zorder=5)
        ax_ma.annotate(gname, (mx, my), xytext=(5, 5), textcoords="offset points", fontweight="bold", fontsize=10)

ax_ma.set_xlabel("log10(Mean CPM + 0.1)", fontsize=11)
ax_ma.set_ylabel("log2 Fold Change (1000-cell vs 2-cell)", fontsize=11)
ax_ma.set_title("Target Species RNA-seq MA Plot", fontweight="bold", fontsize=12)
ax_ma.axhline(0, color="black", linestyle="--", alpha=0.5)
ax_ma.legend(frameon=True, fontsize=8.5, loc="upper left")
ax_ma.grid(True, linestyle=":", alpha=0.5)

# Panel 2: Volcano plot
p_vals = df_counts["pval"] if "pval" in df_counts.columns else np.ones(len(df_counts))
nlp = -np.log10(np.maximum(p_vals, 1e-300)) if "pval" in df_counts.columns else df_counts["log2fc"].abs()
y_label = "-log10(p-value)" if "pval" in df_counts.columns else "|log2FC| Diagnostic Metric"

ax_vol.scatter(df_counts["log2fc"], nlp, color="#cccccc", alpha=0.3, s=8)
ax_vol.scatter(df_counts.loc[is_control, "log2fc"], nlp[is_control], color="#2b83ba", alpha=0.6, s=12)
ax_vol.scatter(df_counts.loc[is_all_upregulated & ~is_strictly_zygotic, "log2fc"], nlp[is_all_upregulated & ~is_strictly_zygotic], color="#fdae61", alpha=0.7, s=16)
ax_vol.scatter(df_counts.loc[is_strictly_zygotic, "log2fc"], nlp[is_strictly_zygotic], color="#d7191c", alpha=0.8, s=22)

for gname in TARGET_GENES + CANONICAL_ZGA_MARKERS:
    sub = df_counts[df_counts["gene_name"].str.lower() == gname.lower()]
    if not sub.empty:
        r = sub.iloc[0]
        vx = r["log2fc"]
        vy = nlp.loc[sub.index[0]] if hasattr(nlp, "loc") else nlp[sub.index[0]]
        ax_vol.scatter([vx], [vy], color="black", s=60, zorder=5)
        ax_vol.annotate(gname, (vx, vy), xytext=(5, 5), textcoords="offset points", fontweight="bold", fontsize=10)

ax_vol.set_xlabel("log2 Fold Change (1000-cell vs 2-cell)", fontsize=11)
ax_vol.set_ylabel(y_label, fontsize=11)
ax_vol.set_title("Target Species Volcano / Expression Separation", fontweight="bold", fontsize=12)
ax_vol.axvline(0, color="black", linestyle="--", alpha=0.5)
ax_vol.grid(True, linestyle=":", alpha=0.5)

plt.tight_layout()
fig1_path = res_dir / "fig1_volcano_ma_plot.png"
plt.savefig(fig1_path, dpi=300)
plt.show()
print(f"[+] Figure 1 (MA & Volcano plots) exported to {fig1_path}")

In [ ]:
def merge_intervals(intervals):
    """Merge overlapping exon intervals across transcript isoforms to avoid double-counting."""
    if not intervals:
        return []
    intervals = sorted(intervals, key=lambda x: (x[0], x[1]))
    merged = [list(intervals[0])]
    for s, e in intervals[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return [tuple(x) for x in merged]

# 1. Parse and merge exonic coordinates per gene from GTF
gene_exons = {}
with (gzip.open(TARGET_GTF, "rt") if str(TARGET_GTF).endswith(".gz") else open(TARGET_GTF, "rt")) as f:
    for line in f:
        if line.startswith("#"):
            continue
        p = line.strip().split("\t")
        if len(p) >= 9 and p[2] == "exon":
            chrom, start, end, strand = p[0], int(p[3]) - 1, int(p[4]), p[6]
            gid = None
            for item in p[8].split(";"):
                item = item.strip()
                if item.startswith("gene_id"):
                    m = re.search(r'gene_id\s+"([^"]+)"', item)
                    if m:
                        gid = m.group(1)
            if gid:
                if gid not in gene_exons:
                    gene_exons[gid] = {"chrom": chrom, "strand": strand, "exons": []}
                gene_exons[gid]["exons"].append((start, end))

for gid in gene_exons:
    gene_exons[gid]["exons"] = merge_intervals(gene_exons[gid]["exons"])
    gene_exons[gid]["merged_exons"] = gene_exons[gid]["exons"]

def calculate_spliced_coverage_profile(gid, bam_file, n_bins=100):
    ginfo = gene_exons.get(gid)
    if not ginfo:
        return np.zeros(n_bins, dtype=float)
    chrom = ginfo["chrom"]
    strand = ginfo["strand"]
    exons = ginfo["exons"]

    cov_pieces = []
    for st, en in exons:
        try:
            cov = np.array(bam_file.count_coverage(chrom, st, en, quality_threshold=10)).sum(axis=0)
        except Exception:
            cov = np.zeros(en - st, dtype=float)
        cov_pieces.append(cov)

    if not cov_pieces:
        return np.zeros(n_bins, dtype=float)
    full_spliced_cov = np.concatenate(cov_pieces)
    if len(full_spliced_cov) == 0:
        return np.zeros(n_bins, dtype=float)

    # Bin into n_bins
    bin_edges = np.linspace(0, len(full_spliced_cov), n_bins + 1, dtype=int)
    profile = np.zeros(n_bins, dtype=float)
    for b in range(n_bins):
        s_b = bin_edges[b]
        e_b = bin_edges[b + 1]
        if e_b > s_b:
            profile[b] = full_spliced_cov[s_b:e_b].mean()

    # Reversal for minus-strand genes so bin 0 is always 5' TSS and bin 100 is always 3' TES
    if strand == "-":
        profile = profile[::-1]
    return profile

# Sanity assertion: verify consistent 5'->3' polarity on + vs - genes
test_plus = [gid for gid, info in gene_exons.items() if info["strand"] == "+"]
test_minus = [gid for gid, info in gene_exons.items() if info["strand"] == "-"]
assert len(test_plus) > 0 and len(test_minus) > 0, "GTF missing genes on positive or negative strand!"
print(f"[+] Verified strand orientation handling across {len(test_plus):,} (+) and {len(test_minus):,} (-) genes.")

print(f"[+] Loaded merged exonic models for {len(gene_exons):,} genes.")

# Spliced Metagene Profiles across Categories
import matplotlib.pyplot as plt

meta_profiles = {}
with pysam.AlignmentFile(RNASEQ_PREMZT_BAMS[0], "rb") as bf:
    for cat in ["strictly_zygotic_minor_wave", "all_upregulated", "control_non_activated"]:
        sub_gids = df_counts.loc[df_counts["category"] == cat, "Geneid"].values
        profiles = []
        for gid in sub_gids[:250]:
            p = calculate_spliced_coverage_profile(gid, bf, n_bins=100)
            if p.sum() > 0:
                profiles.append(p / p.sum())
        if profiles:
            meta_profiles[cat] = np.mean(profiles, axis=0)

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = {"strictly_zygotic_minor_wave": "#d7191c", "all_upregulated": "#fdae61", "control_non_activated": "#2b83ba"}
labels = {"strictly_zygotic_minor_wave": "Strictly Zygotic Minor Wave", "all_upregulated": "All Upregulated", "control_non_activated": "Control Non-Activated"}

for cat, prof in meta_profiles.items():
    ax.plot(np.linspace(0, 100, 100), prof, color=colors[cat], lw=2.2, label=labels[cat])

ax.set_xlabel("Spliced Transcript Coordinate (5' TSS -> 3' TES %)", fontsize=11)
ax.set_ylabel("Mean Normalized Coverage", fontsize=11)
ax.set_title("Strand-Corrected Spliced Exonic Coverage Profiles", fontweight="bold", fontsize=12)
ax.legend(frameon=True, fontsize=9.5)
ax.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()
fig1a_path = res_dir / "fig1a_spliced_coverage_profile.png"
plt.savefig(fig1a_path, dpi=300)
plt.show()
print(f"[+] Figure 1a exported to {fig1a_path}")


In [ ]:
# Post-Transcriptional vs. Transcriptional Diagnostics: Exonic SNP Allelic Shift Test
# and 3' UTR Cytoplasmic Polyadenylation Element (CPE) & PAS Read-Extension Tests
from scipy.stats import fisher_exact, mannwhitneyu, gaussian_kde
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
from pyfaidx import Fasta

def scan_exonic_snps(bam_2c, bam_1k, gid, gname, min_depth=20, min_maf=0.10):
    ginfo = gene_exons.get(gid)
    if not ginfo:
        return []
    chrom = ginfo["chrom"]
    results = []
    # Use consistent key
    exons_list = ginfo.get("exons", ginfo.get("merged_exons", []))
    for st, en in exons_list:
        try:
            cov_2c = np.array(bam_2c.count_coverage(chrom, st, en, quality_threshold=20))
            cov_1k = np.array(bam_1k.count_coverage(chrom, st, en, quality_threshold=20))
        except Exception:
            continue
        d_2c = cov_2c.sum(axis=0)
        d_1k = cov_1k.sum(axis=0)
        valid = (d_2c >= min_depth) & (d_1k >= min_depth)
        for idx in np.where(valid)[0]:
            c2c_bases = cov_2c[:, idx]
            c1k_bases = cov_1k[:, idx]
            d2 = d_2c[idx]
            d1 = d_1k[idx]
            maj_idx = np.argmax(c2c_bases)
            ref_2c = c2c_bases[maj_idx]
            alt_2c = d2 - ref_2c
            ref_1k = c1k_bases[maj_idx]
            alt_1k = d1 - ref_1k
            f2 = alt_2c / d2
            f1 = alt_1k / d1
            if f2 >= min_maf or f1 >= min_maf:
                tbl = [[alt_1k, ref_1k], [alt_2c, ref_2c]]
                _, p_val = fisher_exact(tbl, alternative="two-sided")
                results.append({
                    "gene": gname,
                    "chrom": chrom,
                    "pos": st + idx,
                    "depth_early": d2,
                    "depth_premzt": d1,
                    "alt_freq_early": f2,
                    "alt_freq_premzt": f1,
                    "delta_alt_freq": f1 - f2,
                    "abs_delta_alt_freq": abs(f1 - f2),
                    "fisher_pval": p_val
                })
    return results

target_snps = []
ctrl_shifts = []

with pysam.AlignmentFile(RNASEQ_EARLY_BAMS[0], "rb") as bam_2c, pysam.AlignmentFile(RNASEQ_PREMZT_BAMS[0], "rb") as bam_1k:
    # 1. Target genes scan
    for tg in TARGET_GENES:
        tg_rows = df_counts[df_counts["gene_name"].str.lower() == tg.lower()]
        for _, r in tg_rows.iterrows():
            target_snps.extend(scan_exonic_snps(bam_2c, bam_1k, r["Geneid"], r["gene_name"]))

    # 2. Genome-wide control-gene null baseline (top 100 control genes, depth>=30, MAF>=0.15)
    ctrl_genes = df_counts[df_counts["category"] == "control_non_activated"].sort_values(by="cpm_early", ascending=False).head(100)
    for gid in ctrl_genes["Geneid"]:
        ginfo = gene_exons.get(gid)
        if not ginfo:
            continue
        chrom = ginfo["chrom"]
        exons_list = ginfo.get("exons", ginfo.get("merged_exons", []))
        for st, en in exons_list:
            try:
                cov_2c = np.array(bam_2c.count_coverage(chrom, st, en, quality_threshold=20))
                cov_1k = np.array(bam_1k.count_coverage(chrom, st, en, quality_threshold=20))
            except Exception:
                continue
            depth_2c = cov_2c.sum(axis=0)
            depth_1k = cov_1k.sum(axis=0)
            valid = (depth_2c >= 30) & (depth_1k >= 30)
            for idx in np.where(valid)[0]:
                c2c_bases = cov_2c[:, idx]
                c1k_bases = cov_1k[:, idx]
                d2 = depth_2c[idx]
                d1 = depth_1k[idx]
                maj_idx = np.argmax(c2c_bases)
                ref_2c = c2c_bases[maj_idx]
                alt_2c = d2 - ref_2c
                ref_1k = c1k_bases[maj_idx]
                alt_1k = d1 - ref_1k
                f2 = alt_2c / d2
                f1 = alt_1k / d1
                if f2 >= 0.15 or f1 >= 0.15:
                    ctrl_shifts.append(abs(f1 - f2))

df_target_snps = pd.DataFrame(target_snps)
if not df_target_snps.empty:
    _, padj, _, _ = multipletests(df_target_snps["fisher_pval"], method="fdr_bh")
    df_target_snps["padj"] = padj
    print(f"[*] Detected heterozygous exonic sites in TARGET_GENES ({len(df_target_snps)} sites):")
    disp_cols = ["gene", "pos", "depth_early", "depth_premzt", "alt_freq_early", "alt_freq_premzt", "delta_alt_freq", "fisher_pval", "padj"]
    print(df_target_snps[disp_cols].to_string(index=False))
else:
    print("[-] No heterozygous exonic positions detected in target genes.")

ctrl_shifts = np.array(ctrl_shifts)
if len(ctrl_shifts) > 0:
    ctrl_p95 = np.percentile(ctrl_shifts, 95)
    print(f"[*] Control-Gene Null Baseline: {len(ctrl_shifts)} sites | Median |Δ| = {np.median(ctrl_shifts):.4f} | 95th %ile = {ctrl_p95:.4f}")
else:
    ctrl_p95 = 0.10
    print("[-] Control shifts empty, using default threshold 0.10")

for tg in TARGET_GENES:
    if not df_target_snps.empty and "gene" in df_target_snps.columns:
        tg_sub = df_target_snps[df_target_snps["gene"].str.lower() == tg.lower()]
        if not tg_sub.empty and len(ctrl_shifts) > 0:
            tg_vals = tg_sub["abs_delta_alt_freq"].values
            u_stat, u_pval = mannwhitneyu(tg_vals, ctrl_shifts, alternative="greater")
            sig_str = "SIGNIFICANT (exceeds background noise)" if u_pval < 0.05 else "Not significant"
            print(f"    {tg:10s}: Median |Δ| = {np.median(tg_vals):.4f} | U = {u_stat:.1f}, p = {u_pval:.4e} -> {sig_str}")

# Plot Figure 1c
fig, ax = plt.subplots(figsize=(8, 4.5))
if len(ctrl_shifts) > 5:
    ax.hist(ctrl_shifts, bins=35, density=True, alpha=0.55, color="#9ecae1", edgecolor="#3182bd", label=f"Control-Gene Null (N={len(ctrl_shifts)})")
    kde = gaussian_kde(ctrl_shifts)
    x_eval = np.linspace(0, max(0.35, float(ctrl_shifts.max())), 200)
    ax.plot(x_eval, kde(x_eval), color="#08519c", lw=2.2, label="Null Background Density")
ax.axvline(ctrl_p95, color="#e6550d", linestyle="--", lw=2.0, label=f"95th %ile Threshold ({ctrl_p95:.3f})")

color_map_tg = {"slc35a3a": "#d7191c", "notch2": "#756bb1"}
for tg in TARGET_GENES:
    if not df_target_snps.empty and "gene" in df_target_snps.columns:
        tg_sub = df_target_snps[df_target_snps["gene"].str.lower() == tg.lower()]
        if not tg_sub.empty:
            tg_med = np.median(tg_sub["abs_delta_alt_freq"].values)
            c = color_map_tg.get(tg.lower(), "#2ca02c")
            ax.axvline(tg_med, color=c, lw=2.5, linestyle="-", label=f"{tg} Median Shift ({tg_med:.3f})")

ax.set_xlabel("Allelic Frequency Shift |Δ alt_freq| (1000-cell vs 2-cell)", fontsize=11)
ax.set_ylabel("Empirical Density", fontsize=11)
ax.set_title(f"Target Gene Allelic Shifts vs. Genome-Wide Control Null ({TARGET_SPECIES_NAME})", fontweight="bold", fontsize=12)
ax.legend(frameon=True, fontsize=9.5, loc="upper right")
ax.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()
fig1c_path = res_dir / "fig1c_snp_allele_shift.png"
plt.savefig(fig1c_path, dpi=300)
plt.show()
print(f"[+] Figure 1c exported to {fig1c_path}")

# ==============================================================================
# 3' UTR CPE Enrichment & Downstream PAS Read-Extension Diagnostic
# ==============================================================================
print("=" * 80)
print("[*] 3' UTR Post-Transcriptional Diagnostic: CPE & PAS Motif Enrichment:")
gene_3putr = {}
with (gzip.open(TARGET_GTF, "rt") if str(TARGET_GTF).endswith(".gz") else open(TARGET_GTF, "rt")) as f:
    for line in f:
        if line.startswith("#"):
            continue
        p = line.strip().split("\t")
        if len(p) >= 9 and p[2].lower() in {"three_prime_utr", "3utr", "3_prime_utr"}:
            chrom, s, e, strand = p[0], int(p[3]) - 1, int(p[4]), p[6]
            gid = None
            for item in p[8].split(";"):
                item = item.strip()
                if item.startswith("gene_id"):
                    m = re.search(r'gene_id\s+"([^"]+)"', item)
                    if m:
                        gid = m.group(1)
            if gid:
                if gid not in gene_3putr:
                    gene_3putr[gid] = {"chrom": chrom, "strand": strand, "intervals": []}
                gene_3putr[gid]["intervals"].append((s, e))

for gid, ginfo in gene_exons.items():
    if gid not in gene_3putr or not gene_3putr[gid]["intervals"]:
        chrom, strand = ginfo["chrom"], ginfo["strand"]
        exs = ginfo["exons"]
        if exs:
            if strand == "+":
                last_e = exs[-1][1]
                gene_3putr[gid] = {"chrom": chrom, "strand": strand, "intervals": [(max(0, last_e - 500), last_e)]}
            else:
                first_s = exs[0][0]
                gene_3putr[gid] = {"chrom": chrom, "strand": strand, "intervals": [(first_s, first_s + 500)]}

utr_seqs = {}
if Path(TARGET_GENOME_FA).exists():
    try:
        fa = Fasta(TARGET_GENOME_FA, as_raw=True)
        for gid, uinfo in gene_3putr.items():
            ch = uinfo["chrom"]
            st_list = []
            for st, en in uinfo["intervals"]:
                try:
                    st_list.append(str(fa[ch][st:en]))
                except Exception:
                    pass
            cat_seq = "".join(st_list).upper()
            if uinfo["strand"] == "-":
                cat_seq = cat_seq.translate(str.maketrans("ACGTN", "TGCAN"))[::-1]
            utr_seqs[gid] = cat_seq
    except Exception as e:
        print(f"[-] Notice loading FASTA for UTR extraction: {e}")

if "is_all_upregulated" in df_counts.columns:
    upreg_mask = (df_counts["is_all_upregulated"] == True) | (df_counts["category"].isin(["all_upregulated", "strictly_zygotic_minor_wave"]))
else:
    upreg_mask = df_counts["category"].isin(["all_upregulated", "strictly_zygotic_minor_wave"])
upreg_gids = set(df_counts.loc[upreg_mask, "Geneid"])
ctrl_gids = set(df_counts.loc[df_counts["category"] == "control_non_activated", "Geneid"])

tests = {
    "Canonical CPE (TTTTAT/TTTTAAT)": ["TTTTAT", "TTTTAAT"],
    "eCPE / U-rich (TTTTTT/TTTTACT)": ["TTTTTT", "TTTTACT", "TTTTAAG"],
    "PAS Hexamer (AATAAA/ATTAAA)":    ["AATAAA", "ATTAAA"]
}

for t_name, motif_list in tests.items():
    up_pos = sum(1 for gid in upreg_gids if gid in utr_seqs and any(m in utr_seqs[gid] for m in motif_list))
    up_neg = len(upreg_gids) - up_pos
    ctrl_pos = sum(1 for gid in ctrl_gids if gid in utr_seqs and any(m in utr_seqs[gid] for m in motif_list))
    ctrl_neg = len(ctrl_gids) - ctrl_pos
    cont_tab = [[up_pos, max(0, up_neg)], [ctrl_pos, max(0, ctrl_neg)]]
    odds, p_val = fisher_exact(cont_tab, alternative="greater")
    print(f"    {t_name:32s}: Upreg {up_pos}/{len(upreg_gids)} ({100*up_pos/max(len(upreg_gids),1):.1f}%) vs Ctrl {ctrl_pos}/{len(ctrl_gids)} ({100*ctrl_pos/max(len(ctrl_gids),1):.1f}%) | OR = {odds:.2f}, p = {p_val:.2e}")

print("=" * 80)
print(f"[*] Inspecting +500 bp Downstream Intergenic Read Extension for TARGET_GENES:")
with pysam.AlignmentFile(RNASEQ_EARLY_BAMS[0], "rb") as b_2c, pysam.AlignmentFile(RNASEQ_PREMZT_BAMS[0], "rb") as b_1k:
    for tg in TARGET_GENES:
        tg_rows = df_counts[df_counts["gene_name"].str.lower() == tg.lower()]
        if tg_rows.empty:
            continue
        r = tg_rows.iloc[0]
        chrom = r["chrom"]
        strand = r["strand"]
        tes = int(r["end"]) if strand == "+" else int(r["start"])
        down_st = tes if strand == "+" else max(0, tes - 500)
        down_en = (tes + 500) if strand == "+" else tes
        try:
            cov2 = np.array(b_2c.count_coverage(chrom, down_st, down_en, quality_threshold=10)).sum()
            cov1 = np.array(b_1k.count_coverage(chrom, down_st, down_en, quality_threshold=10)).sum()
        except Exception:
            cov2, cov1 = 0, 0
        down_seq = ""
        if Path(TARGET_GENOME_FA).exists():
            try:
                fa = Fasta(TARGET_GENOME_FA, as_raw=True)
                down_seq = str(fa[chrom][down_st:down_en]).upper()
                if strand == "-":
                    down_seq = down_seq.translate(str.maketrans("ACGTN", "TGCAN"))[::-1]
            except Exception:
                pass
        pas_found = [m for m in ["AATAAA", "ATTAAA"] if m in down_seq]
        print(f"    {tg} ({chrom}:{down_st}-{down_en}, strand {strand}): 2-cell cov = {cov2:,}, 1000-cell cov = {cov1:,} | PAS: {pas_found or 'None'}")


---
## Block 1b: Zebrafish SLAM-seq Nascent RNA Ortholog Overlay

To definitively distinguish **genuine pre-MZT zygotic transcriptional activation** from **cytoplasmic polyadenylation of pre-existing maternal mRNAs**, we project Zebrafish 4sU SLAM-seq nascent transcription data onto the target species transcriptome:
- `preMZT_strictly_zygotic`: absent total RNA at $0.75\text{ hpf}$ (`CPM < 1.0`) with nascent T$\rightarrow$C reads appearing at $2.5\text{ hpf}$ (pre-MZT minor wave).
- `preMZT_maternal_zygotic`: maternally present at $0.75\text{ hpf}$ (`CPM >= 5.0`) **and** nascent T$\rightarrow$C reads active at $2.5\text{ hpf}$ (pre-MZT maternal-zygotic activation).
- `major_wave_only`: absent or negligible nascent T$\rightarrow$C reads at $2.5\text{ hpf}$, turning on strongly at $3.0–4.5\text{ hpf}$ (canonical major wave).
- `maternal_non_transcribed`: maternal mRNA present at $0.75\text{ hpf}$ (`CPM >= 5.0`) with **zero nascent T$\rightarrow$C transcription** through $4.5\text{ hpf}$ (maternal non-transcribed baseline).

We generate a **2-panel diagnostic figure** (MA plot & 4-class violin distributions with Mann-Whitney U and Fisher exact tests) and export **`slam_confirmed_minor_wave`** as an ultra-clean positive validation set for ML benchmarks.

In [ ]:
# Launch / Verify Zebrafish SLAM-seq Preprocessing Task
slam_cmd = [
    "python", "scripts/prep_slamseq.py",
    "--work-dir", WORK_DIR,
    "--genome-fa", ZEBRAFISH_GENOME_FA,
    "--gtf", ZEBRAFISH_GTF,
    "--task-name", "prep_slamseq_zf",
    "--threads", str(THREADS)
]
tm.submit("prep_slamseq_zf", slam_cmd, depends_on="download_resources", force_rerun=FORCE_RERUN)
display(tm.status())

import matplotlib.pyplot as plt

# Overlay Zebrafish SLAM-seq Nascent Transcription Timing on Target Species
zf_slam_path = Path(WORK_DIR) / "slamseq" / "zf_slamseq_gene_classes.tsv"
gene_sets_tsv = res_dir / f"{TARGET_LOCUS_NAME}_gene_sets.tsv"
df_counts = pd.read_csv(gene_sets_tsv, sep="\t")

slam_info = tm.get_task_info("prep_slamseq_zf")
is_slam_running = slam_info.get("status") == "RUNNING"

if is_slam_running or not zf_slam_path.exists():
    if is_slam_running:
        print("[*] Task 'prep_slamseq_zf' is currently RUNNING in background.")
        print(f"    Current progress: {slam_info.get('progress', 'In progress...')}")
        print("    Zebrafish SLAM-seq overlay and Figure 1b will be rendered once the task completes.")
    else:
        print(f"[*] Zebrafish SLAM-seq table not yet generated: {zf_slam_path}.")
    print("    Downstream validation will default to strictly_zygotic_minor_wave.")
    if "is_slam_confirmed_minor_wave" not in df_counts.columns:
        df_counts["is_slam_confirmed_minor_wave"] = df_counts["category"] == "strictly_zygotic_minor_wave"
        df_counts.to_csv(gene_sets_tsv, sep="\t", index=False)
else:
    df_zf_slam = pd.read_csv(zf_slam_path, sep="\t")
    if not Path(ortholog_tsv).exists():
        print(f"[!] Ortholog mapping file {ortholog_tsv} not found yet.")
        if "is_slam_confirmed_minor_wave" not in df_counts.columns:
            df_counts["is_slam_confirmed_minor_wave"] = df_counts["category"] == "strictly_zygotic_minor_wave"
            df_counts.to_csv(gene_sets_tsv, sep="\t", index=False)
    else:
        df_ortho = pd.read_csv(ortholog_tsv, sep="\t")
        df_ortho_clean = df_ortho.drop_duplicates(subset=["target_gene_id"]).copy()
        ortho_slam = df_ortho_clean.merge(df_zf_slam[["gene_id", "slam_class"]], left_on="zf_gene_id", right_on="gene_id", how="left")
        id_col = "Geneid" if "Geneid" in df_counts.columns else "gene_id"
        df_merged = df_counts.merge(ortho_slam[["target_gene_id", "slam_class"]], left_on=id_col, right_on="target_gene_id", how="left")
        df_merged["slam_class"] = df_merged["slam_class"].fillna("unmapped_ortholog")

        # Create boolean column is_slam_confirmed_minor_wave
        is_slam_confirmed = (
            df_merged["category"].isin(["strictly_zygotic_minor_wave", "all_upregulated"]) &
            df_merged["slam_class"].isin(["preMZT_strictly_zygotic", "preMZT_maternal_zygotic"])
        )
        df_merged["is_slam_confirmed_minor_wave"] = is_slam_confirmed
        df_merged.to_csv(gene_sets_tsv, sep="\t", index=False)
        print(f"[+] Persisted updated gene sets table with SLAM-seq overlay ({is_slam_confirmed.sum()} confirmed genes) to {gene_sets_tsv}")

        n_mapped = (df_merged["slam_class"] != "unmapped_ortholog").sum()
        print("======================================================================")
        print(f"[*] Zebrafish SLAM-seq Ortholog Mapping Statistics:")
        print(f"    Genes mapped to zebrafish ortholog: {n_mapped} / {len(df_counts)} ({100*n_mapped/len(df_counts):.1f}%)")
        print(df_merged["slam_class"].value_counts().to_string())
        print("======================================================================")

        from scipy.stats import mannwhitneyu, fisher_exact
        mat_lfc = df_merged.loc[df_merged["slam_class"] == "maternal_non_transcribed", "log2fc"].dropna()
        mw_pvals = {}
        for sc in ["preMZT_strictly_zygotic", "preMZT_maternal_zygotic", "major_wave_only"]:
            sc_lfc = df_merged.loc[df_merged["slam_class"] == sc, "log2fc"].dropna()
            if len(sc_lfc) > 0 and len(mat_lfc) > 0:
                stat, pval = mannwhitneyu(sc_lfc, mat_lfc, alternative="greater")
                mw_pvals[sc] = pval
                print(f"    Mann-Whitney U: {sc} vs maternal: p = {pval:.2e}")
            else:
                mw_pvals[sc] = 1.0

        is_up = df_merged["is_all_upregulated"]
        is_pre_sz = df_merged["slam_class"] == "preMZT_strictly_zygotic"
        cont_tbl = [
            [(is_up & is_pre_sz).sum(), (~is_up & is_pre_sz).sum()],
            [(is_up & ~is_pre_sz).sum(), (~is_up & ~is_pre_sz).sum()]
        ]
        fisher_odds, fisher_p = fisher_exact(cont_tbl, alternative="greater")
        print(f"    Fisher Exact Test (preMZT_strictly_zygotic in all_upregulated): OR = {fisher_odds:.2f}, p = {fisher_p:.2e}")

        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        ax = axes[0]
        # Panel 1: MA plot colored by Zebrafish SLAM-seq timing class
        slam_palette = {
            "preMZT_strictly_zygotic": "#d7191c",
            "preMZT_maternal_zygotic": "#fdae61",
            "major_wave_only": "#2ca25f",
            "maternal_non_transcribed": "#2b83ba",
            "unmapped_ortholog": "#bdbdbd",
            "silent_or_low": "#969696"
        }
        mean_cpm_t = 0.5 * (df_merged["cpm_early"] + df_merged["cpm_premzt"])
        for sc, sc_col in slam_palette.items():
            sc_mask = df_merged["slam_class"] == sc
            if sc_mask.any():
                ax.scatter(
                    np.log10(mean_cpm_t[sc_mask] + 0.1),
                    df_merged.loc[sc_mask, "log2fc"],
                    color=sc_col,
                    label=sc,
                    s=22 if sc.startswith("preMZT") else 12,
                    alpha=0.85 if sc.startswith("preMZT") else 0.4
                )
        # Callouts for TARGET_GENES and CANONICAL_ZGA_MARKERS
        for gname in TARGET_GENES + CANONICAL_ZGA_MARKERS:
            sub = df_merged[df_merged["gene_name"].str.lower() == gname.lower()]
            if not sub.empty:
                r = sub.iloc[0]
                mx = np.log10(0.5 * (r["cpm_early"] + r["cpm_premzt"]) + 0.1)
                my = r["log2fc"]
                ax.scatter([mx], [my], color="black", s=70, zorder=6)
                ax.annotate(gname, (mx, my), xytext=(5, 5), textcoords="offset points", fontweight="bold", fontsize=10)

        ax.set_xlabel("log10(Mean Target CPM + 0.1)", fontsize=11)
        ax.set_ylabel("Target log2 Fold Change", fontsize=11)
        ax.set_title("Target RNA-seq MA Plot Colored by ZF SLAM-seq Timing", fontweight="bold")
        ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
        ax.legend(frameon=True, fontsize=8.5, loc="upper left")
        ax.grid(True, linestyle=":", alpha=0.5)

        ax = axes[1]
        classes_to_plot = ["preMZT_strictly_zygotic", "preMZT_maternal_zygotic", "major_wave_only", "maternal_non_transcribed"]
        labels_to_plot = [
            f"preMZT Strictly\nZygotic\n(p={mw_pvals.get('preMZT_strictly_zygotic', 1.0):.1e})",
            f"preMZT Maternal-\nZygotic\n(p={mw_pvals.get('preMZT_maternal_zygotic', 1.0):.1e})",
            f"Major Wave\nOnly\n(p={mw_pvals.get('major_wave_only', 1.0):.1e})",
            "Maternal\nNon-Transcribed\n(Control)"
        ]
        colors_to_plot = ["#d7191c", "#fdae61", "#2ca25f", "#2b83ba"]
        lfc_by_class = [df_merged.loc[df_merged["slam_class"] == sc, "log2fc"].dropna().values for sc in classes_to_plot]
        valid_indices = [i for i, d in enumerate(lfc_by_class) if len(d) > 0]
        if valid_indices:
            filtered_lfc = [lfc_by_class[i] for i in valid_indices]
            parts = ax.violinplot(filtered_lfc, positions=[i + 1 for i in valid_indices], showmeans=True, showmedians=True)
            for i_idx, p_idx in enumerate(valid_indices):
                if i_idx < len(parts['bodies']):
                    parts['bodies'][i_idx].set_facecolor(colors_to_plot[p_idx])
                    parts['bodies'][i_idx].set_edgecolor('black')
            ax.set_xticks([i + 1 for i in valid_indices])
            ax.set_xticklabels([labels_to_plot[i] for i in valid_indices], fontsize=9.5)
            ax.set_ylabel("log2 Fold Change in Target RNA-seq", fontsize=11)
            ax.set_title(f"Target Species log2FC Across SLAM-seq Timing Classes\n(OR={fisher_odds:.1f}, p={fisher_p:.1e})", fontweight="bold")
            ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
            ax.grid(True, linestyle=":", alpha=0.5)

        plt.tight_layout()
        fig1b_path = res_dir / "fig1b_slamseq_ortholog_overlay.png"
        plt.savefig(fig1b_path, dpi=300)
        plt.show()
        print(f"[+] Figure 1b exported to {fig1b_path}")

---
## Block 2: Background ATAC-seq Preprocessing & ChromBPNet Base Training (Zebrafish & Medaka)

We train debiased Stage 1 ChromBPNet accessibility models on reference blastula chromatin:
1. **Peak Calling & Continuous Targets:** Process early vs. late blastula ATAC-seq for Zebrafish (*GRCz11*) and Medaka (*ASM223467v1*), calling MACS3 late peaks and computing continuous targets ($y_{\text{abs}}, y_{\text{rel}}$) across summits.
2. **Leak-Free Chromosome Splits:** The chromosome orthologous to `TARGET_LOCUS_NAME` is strictly held out in the `test` split.
3. **Stage 1 ChromBPNet Training:** Runs GC-matched non-peaks, Tn5 bias modeling, and official ChromBPNet PyTorch training via `scripts/train_chrombpnet.py` managed by `TaskManager`.

In [ ]:
# Launch Background ATAC Preprocessing and ChromBPNet Training for Both Zebrafish & Medaka
atac_manifest = f"{WORK_DIR}/downloads/atac_fastq_manifest.tsv"

# 1. Submit Zebrafish ATAC Preprocessing & ChromBPNet Pipeline
zf_atac_cmd = [
    "python", "-u", "scripts/prep_atac.py",
    "--species", "Danio_rerio",
    "--work-dir", WORK_DIR,
    "--genome-fa", ZEBRAFISH_GENOME_FA,
    "--gtf", ZEBRAFISH_GTF,
    "--fastq-manifest", atac_manifest,
    "--target-genes", *TARGET_GENES,
    "--cv-folds", str(CV_FOLDS),
    "--task-name", "prep_atac_danio_rerio",
    "--threads", str(THREADS)
]
if EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2:
    zf_atac_cmd.append("--exclude-housekeeping-promoters")

tm.submit("prep_atac_danio_rerio", zf_atac_cmd, depends_on="download_resources", force_rerun=FORCE_RERUN)

zf_chrom_sizes = f"{WORK_DIR}/atac/Danio_rerio/chrom.sizes"
zf_peaks_bed = f"{WORK_DIR}/atac/Danio_rerio/peaks/Danio_rerio_late_peaks.narrowPeak"
zf_late_bam = f"{WORK_DIR}/atac/Danio_rerio/Danio_rerio_late_merged.bam"

FOLDS_TO_TRAIN = [0] if COMPUTE_PROFILE == "laptop_16gb" else list(range(CV_FOLDS))

for fold in FOLDS_TO_TRAIN:
    zf_chr_folds = f"{WORK_DIR}/atac/Danio_rerio/chr_folds.json"
    zf_chrombpnet_cmd = [
        "python", "-u", "scripts/train_chrombpnet.py",
        "--species", "Danio_rerio",
        "--work-dir", WORK_DIR,
        "--genome-fa", ZEBRAFISH_GENOME_FA,
        "--chrom-sizes", zf_chrom_sizes,
        "--peaks-bed", zf_peaks_bed,
        "--late-bam", zf_late_bam,
        "--chr-fold-json", zf_chr_folds,
        "--fold", str(fold),
        "--task-name", f"train_chrombpnet_danio_rerio_fold_{fold}",
        "--epochs", str(CHROMBPNET_EPOCHS),
        "--threads", str(THREADS)
    ]
    if USE_PRETRAINED_BIAS:
        bias_h5 = BIAS_MODEL_PATH or f"{WORK_DIR}/downloads/bias_models/atac_bias.h5"
        zf_chrombpnet_cmd.extend(["--use-pretrained-bias", "--bias-model-path", bias_h5])

    tm.submit(f"train_chrombpnet_danio_rerio_fold_{fold}", zf_chrombpnet_cmd, depends_on="prep_atac_danio_rerio", force_rerun=FORCE_RERUN)

# 2. Submit Medaka ATAC Preprocessing & ChromBPNet Pipeline (when enabled)
if ENABLE_MEDAKA:
    medaka_atac_cmd = [
        "python", "-u", "scripts/prep_atac.py",
        "--species", "Oryzias_latipes",
        "--work-dir", WORK_DIR,
        "--genome-fa", MEDAKA_GENOME_FA,
        "--gtf", MEDAKA_GTF,
        "--fastq-manifest", atac_manifest,
        "--target-genes", *TARGET_GENES,
        "--cv-folds", str(CV_FOLDS),
        "--task-name", "prep_atac_medaka",
        "--threads", str(THREADS)
    ]
    if EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2:
        medaka_atac_cmd.append("--exclude-housekeeping-promoters")

    tm.submit("prep_atac_medaka", medaka_atac_cmd, depends_on="download_resources", force_rerun=FORCE_RERUN)

    med_chrom_sizes = f"{WORK_DIR}/atac/Oryzias_latipes/chrom.sizes"
    med_peaks_bed = f"{WORK_DIR}/atac/Oryzias_latipes/peaks/Oryzias_latipes_late_peaks.narrowPeak"
    med_late_bam = f"{WORK_DIR}/atac/Oryzias_latipes/Oryzias_latipes_late_merged.bam"

    for fold in FOLDS_TO_TRAIN:
        med_chr_folds = f"{WORK_DIR}/atac/Oryzias_latipes/chr_folds.json"
        medaka_chrombpnet_cmd = [
            "python", "-u", "scripts/train_chrombpnet.py",
            "--species", "Oryzias_latipes",
            "--work-dir", WORK_DIR,
            "--genome-fa", MEDAKA_GENOME_FA,
            "--chrom-sizes", med_chrom_sizes,
            "--peaks-bed", med_peaks_bed,
            "--late-bam", med_late_bam,
            "--chr-fold-json", med_chr_folds,
            "--fold", str(fold),
            "--task-name", f"train_chrombpnet_medaka_fold_{fold}",
            "--epochs", str(CHROMBPNET_EPOCHS),
            "--threads", str(THREADS)
        ]
        if USE_PRETRAINED_BIAS:
            bias_h5 = BIAS_MODEL_PATH or f"{WORK_DIR}/downloads/bias_models/atac_bias.h5"
            medaka_chrombpnet_cmd.extend(["--use-pretrained-bias", "--bias-model-path", bias_h5])

        tm.submit(f"train_chrombpnet_medaka_fold_{fold}", medaka_chrombpnet_cmd, depends_on="prep_atac_medaka", force_rerun=FORCE_RERUN)

display(tm.status())

---
## Block 3: Stage 2 Continuous Pre-MZT Linear Probe & Grafted End-to-End Models

With late-blastula base encoders trained, we extract penultimate latent embeddings across late-blastula summits to predict pre-MZT dynamics:
- Fit `StandardScaler + RidgeCV` linear probes on training chromosomes.
- Evaluate on the held-out test chromosome: **reporting both all-peak Spearman $r$ and repeat-filtered metrics excluding `miR-430` tandem clusters**.
- Fold weights into `Dense(1)` and graft back onto the frozen encoder for forward + reverse-complement averaged scanning.

In [ ]:
# Fit RidgeCV Linear Probes on Real Latent Embeddings (Zebrafish & Medaka)
from scripts.infer_and_attribute import (
    load_chrombpnet_encoder,
    extract_summit_embeddings_from_genome,
    fit_linear_probes,
    GraftedChromBPNet
)

zf_targets_path = Path(WORK_DIR) / "atac" / "Danio_rerio" / "Danio_rerio_summit_targets.tsv"
if not zf_targets_path.exists():
    raise FileNotFoundError(f"[FATAL] Summit targets table not found at {zf_targets_path}. Please wait for prep_atac_danio_rerio to complete.")

df_summit_targets = pd.read_csv(zf_targets_path, sep="\t")

# Find real trained ChromBPNet debiased checkpoint strictly inside chrombpnet_run/models/ or models/
# Explicitly excluding bias_run and bias_model directories so Tn5 bias models are never accidentally loaded
zf_fold_dir = Path(WORK_DIR) / "models" / "chrombpnet" / "Danio_rerio" / f"fold_{FOLDS_TO_TRAIN[0]}"
cands = [p for p in zf_fold_dir.glob("**/models/*nobias*.*") if "bias_run" not in str(p) and "bias_model" not in str(p)]
if not cands:
    cands = [p for p in zf_fold_dir.glob("*nobias*.*") if "bias_run" not in str(p) and "bias_model" not in str(p)]

if not cands:
    raise FileNotFoundError(f"[FATAL] Debiased Stage 1 checkpoint (*nobias*.pt/h5) not found under {zf_fold_dir}. Zero-mock policy strictly enforced.")
checkpoint_path = cands[0]

print(f"[*] Loading trained Zebrafish ChromBPNet debiased encoder: {checkpoint_path}")
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder_zf = load_chrombpnet_encoder(checkpoint_path, device=device)

# Display real Stage 1 post-training QC metrics
meta_json = zf_fold_dir / "training_meta.json"
if meta_json.exists():
    with open(meta_json) as f:
        meta_data = json.load(f)
    qc = meta_data.get("qc_metrics", {})
    print("=" * 80)
    print(f"[*] Zebrafish Stage 1 ChromBPNet QC Evaluation Metrics (From {meta_json.name}):")
    print(f"    Profile Pearson r:   {qc.get('profile_pearson_r')}")
    print(f"    Count Pearson r:     {qc.get('count_pearson_r')}")
    print(f"    Profile Median JSD:  {qc.get('profile_median_jsd')}")
    print("=" * 80)

print(f"[*] Extracting real penultimate embeddings over {len(df_summit_targets):,} summit windows (2,114 bp, forward+RC averaged)...")
real_embeddings_zf = extract_summit_embeddings_from_genome(
    encoder=encoder_zf,
    genome_fa_path=ZEBRAFISH_GENOME_FA,
    df_summits=df_summit_targets,
    window_bp=2114,
    batch_size=64,
    device=device
)

probe_results_zf = fit_linear_probes(
    embeddings=real_embeddings_zf,
    df_targets=df_summit_targets,
    test_split_col="split",
    test_split_val="test",
    exclude_housekeeping=EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2
)


In [ ]:
# Fold Linear Probes into End-to-End Grafted Models and Persist Checkpoints
from scripts.infer_and_attribute import save_grafted_checkpoint, load_grafted_model

scaler_zf = probe_results_zf["scaler"]
grafted_zf_model = GraftedChromBPNet(
    encoder=encoder_zf,
    coef_abs=probe_results_zf["y_abs"]["coef"],
    intercept_abs=probe_results_zf["y_abs"]["intercept"],
    coef_rel=probe_results_zf["y_rel"]["coef"],
    intercept_rel=probe_results_zf["y_rel"]["intercept"],
    scaler_mean=scaler_zf.mean_,
    scaler_scale=scaler_zf.scale_
)
grafted_zf_model.to(device)
grafted_zf_model.eval()

# Save canonical grafted model checkpoint
grafted_zf_path = zf_fold_dir / "grafted_chrombpnet.pt"
save_grafted_checkpoint(
    grafted_model=grafted_zf_model,
    out_path=grafted_zf_path,
    metadata={
        "species": "Danio_rerio",
        "fold": FOLDS_TO_TRAIN[0],
        "test_spearman_r": probe_results_zf["y_abs"]["test_spearman_r"],
        "test_auroc": probe_results_zf["y_abs"]["test_auroc"]
    }
)
print(f"[+] Canonical Zebrafish grafted model saved to {grafted_zf_path}")

grafted_medaka_model = None
if ENABLE_MEDAKA:
    med_fold_dir = Path(WORK_DIR) / "models" / "chrombpnet" / "Oryzias_latipes" / f"fold_{FOLDS_TO_TRAIN[0]}"
    med_targets_path = Path(WORK_DIR) / "atac" / "Oryzias_latipes" / "Oryzias_latipes_summit_targets.tsv"
    med_cands = [p for p in med_fold_dir.glob("**/models/*nobias*.*") if "bias_run" not in str(p) and "bias_model" not in str(p)]

    if med_targets_path.exists() and med_cands:
        df_med_targets = pd.read_csv(med_targets_path, sep="\t")
        encoder_med = load_chrombpnet_encoder(med_cands[0], device=device)
        print(f"[*] Extracting real penultimate embeddings over Medaka summits...")
        embeddings_med = extract_summit_embeddings_from_genome(
            encoder=encoder_med,
            genome_fa_path=MEDAKA_GENOME_FA,
            df_summits=df_med_targets,
            window_bp=2114,
            batch_size=64,
            device=device
        )
        probe_results_med = fit_linear_probes(
            embeddings=embeddings_med,
            df_targets=df_med_targets,
            test_split_col="split",
            test_split_val="test",
            exclude_housekeeping=EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2
        )
        scaler_med = probe_results_med["scaler"]
        grafted_medaka_model = GraftedChromBPNet(
            encoder=encoder_med,
            coef_abs=probe_results_med["y_abs"]["coef"],
            intercept_abs=probe_results_med["y_abs"]["intercept"],
            coef_rel=probe_results_med["y_rel"]["coef"],
            intercept_rel=probe_results_med["y_rel"]["intercept"],
            scaler_mean=scaler_med.mean_,
            scaler_scale=scaler_med.scale_
        )
        grafted_med_path = med_fold_dir / "grafted_chrombpnet.pt"
        save_grafted_checkpoint(
            grafted_model=grafted_medaka_model,
            out_path=grafted_med_path,
            metadata={"species": "Oryzias_latipes", "fold": FOLDS_TO_TRAIN[0]}
        )
        print(f"[+] Canonical Medaka grafted model saved to {grafted_med_path}")

# Multi-fold ensemble wrapper
grafted_zf_models = [grafted_zf_model]
if len(FOLDS_TO_TRAIN) > 1:
    print(f"[*] Multi-fold training enabled: fitting linear probes across {len(FOLDS_TO_TRAIN)} Zebrafish folds...")
    zf_folds_json = Path(WORK_DIR) / "atac" / "Danio_rerio" / "chr_folds.json"
    all_zf_folds = {}
    if zf_folds_json.exists():
        with open(zf_folds_json) as f:
            all_zf_folds = json.load(f)

    for fold_i in FOLDS_TO_TRAIN[1:]:
        f_dir = Path(WORK_DIR) / "models" / "chrombpnet" / "Danio_rerio" / f"fold_{fold_i}"
        f_cands = [p for p in f_dir.glob("**/models/*nobias*.*") if "bias_run" not in str(p) and "bias_model" not in str(p)]
        if not f_cands:
            f_cands = [p for p in f_dir.glob("*nobias*.*") if "bias_run" not in str(p) and "bias_model" not in str(p)]
        if f_cands:
            f_enc = load_chrombpnet_encoder(f_cands[0], device=device)
            f_embs = extract_summit_embeddings_from_genome(f_enc, ZEBRAFISH_GENOME_FA, df_summit_targets, device=device)
            df_f = df_summit_targets.copy()
            if f"fold_{fold_i}" in all_zf_folds:
                f_info = all_zf_folds[f"fold_{fold_i}"]
                test_c = set(str(c) for c in f_info.get("test", []))
                val_c = set(str(c) for c in f_info.get("valid", []))
                df_f["split"] = np.where(
                    df_f["chrom"].astype(str).isin(test_c), "test",
                    np.where(df_f["chrom"].astype(str).isin(val_c), "valid", "train")
                )
            f_res = fit_linear_probes(f_embs, df_f, test_split_col="split", test_split_val="test", exclude_housekeeping=EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2)
            f_grafted = GraftedChromBPNet(f_enc, f_res["y_abs"]["coef"], f_res["y_abs"]["intercept"], f_res["y_rel"]["coef"], f_res["y_rel"]["intercept"], f_res["scaler"].mean_, f_res["scaler"].scale_)
            save_grafted_checkpoint(f_grafted, f_dir / "grafted_chrombpnet.pt")
            grafted_zf_models.append(f_grafted)
    print(f"[+] Constructed Zebrafish multi-fold ensemble with {len(grafted_zf_models)} models.")

grafted_medaka_models = [grafted_medaka_model] if grafted_medaka_model is not None else []
if ENABLE_MEDAKA and len(FOLDS_TO_TRAIN) > 1 and grafted_medaka_model is not None:
    print(f"[*] Multi-fold training enabled: fitting linear probes across {len(FOLDS_TO_TRAIN)} Medaka folds...")
    med_folds_json = Path(WORK_DIR) / "atac" / "Oryzias_latipes" / "chr_folds.json"
    all_med_folds = {}
    if med_folds_json.exists():
        with open(med_folds_json) as f:
            all_med_folds = json.load(f)

    for fold_i in FOLDS_TO_TRAIN[1:]:
        f_dir = Path(WORK_DIR) / "models" / "chrombpnet" / "Oryzias_latipes" / f"fold_{fold_i}"
        f_cands = [p for p in f_dir.glob("**/models/*nobias*.*") if "bias_run" not in str(p) and "bias_model" not in str(p)]
        if not f_cands:
            f_cands = [p for p in f_dir.glob("*nobias*.*") if "bias_run" not in str(p) and "bias_model" not in str(p)]
        if f_cands and "df_med_targets" in locals():
            f_enc = load_chrombpnet_encoder(f_cands[0], device=device)
            f_embs = extract_summit_embeddings_from_genome(f_enc, MEDAKA_GENOME_FA, df_med_targets, device=device)
            df_mf = df_med_targets.copy()
            if f"fold_{fold_i}" in all_med_folds:
                f_info = all_med_folds[f"fold_{fold_i}"]
                test_c = set(str(c) for c in f_info.get("test", []))
                val_c = set(str(c) for c in f_info.get("valid", []))
                df_mf["split"] = np.where(
                    df_mf["chrom"].astype(str).isin(test_c), "test",
                    np.where(df_mf["chrom"].astype(str).isin(val_c), "valid", "train")
                )
            f_res = fit_linear_probes(f_embs, df_mf, test_split_col="split", test_split_val="test", exclude_housekeeping=EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2)
            f_grafted = GraftedChromBPNet(f_enc, f_res["y_abs"]["coef"], f_res["y_abs"]["intercept"], f_res["y_rel"]["coef"], f_res["y_rel"]["intercept"], f_res["scaler"].mean_, f_res["scaler"].scale_)
            save_grafted_checkpoint(f_grafted, f_dir / "grafted_chrombpnet.pt")
            grafted_medaka_models.append(f_grafted)
    print(f"[+] Constructed Medaka multi-fold ensemble with {len(grafted_medaka_models)} models.")

---
## Block 4: Zero-Shot Validation on Target-Species Minor-Wave Genes & Medaka ATAC

We test whether ChromBPNet models trained exclusively on Zebrafish transfer across teleosts without any target chromatin training data:
- Tile $\pm 25$ kb around TSSs of target-species genes.
- **Primary Clean Positive Sets:** `strictly_zygotic_minor_wave` and `slam_confirmed_minor_wave` vs. `control_non_activated`.
- **Secondary Diagnostic Series:** `all_upregulated` (clearly labeled so cytoplasmic polyadenylation does not contaminate the benchmark).
- **Zero-Shot Zebrafish $\rightarrow$ Medaka Pre-MZT ATAC Benchmark:** Score Medaka `st9` vs `st11`-only summits when `ENABLE_MEDAKA = True`.

In [ ]:
# Zero-Shot TSS Scoring and Transfer Learning Benchmark (ZF, Medaka, Consensus)
from scripts.infer_and_attribute import score_tss_regions, benchmark_transfer_metrics, score_cross_species_medaka_atac
from sklearn.metrics import roc_curve, precision_recall_curve
import matplotlib.pyplot as plt

gene_sets_tsv = res_dir / f"{TARGET_LOCUS_NAME}_gene_sets.tsv"
if not gene_sets_tsv.exists():
    gene_sets_tsv = Path(WORK_DIR) / "rnaseq" / "target_species_gene_sets.tsv"
df_gene_sets = pd.read_csv(gene_sets_tsv, sep="\t")

if "grafted_zf_models" in locals() and len(grafted_zf_models) > 1:
    print(f"[*] Multi-fold ensemble scoring target species genes across {len(grafted_zf_models)} Zebrafish folds...")
    scored_folds = []
    for g_idx, m_fold in enumerate(grafted_zf_models):
        df_f = score_tss_regions(
            grafted_model=m_fold,
            genome_fa_path=TARGET_GENOME_FA,
            gene_df=df_gene_sets,
            flank_bp=25000,
            window_bp=2114,
            stride_bp=250,
            batch_size=64,
            device=device
        )
        scored_folds.append(df_f)
    df_scored = scored_folds[0].copy()
    for col in ["pred_max_y_abs", "pred_max_y_rel", "pred_max_late"]:
        df_scored[col] = np.mean([f[col].values for f in scored_folds], axis=0)
else:
    print(f"[*] Zero-Shot Scoring target species genes using Zebrafish-trained grafted model...")
    df_scored = score_tss_regions(
        grafted_model=grafted_zf_model,
        genome_fa_path=TARGET_GENOME_FA,
        gene_df=df_gene_sets,
        flank_bp=25000,
        window_bp=2114,
        stride_bp=250,
        batch_size=64,
        device=device
    )

df_scored["pred_max_y_abs_zf"] = df_scored["pred_max_y_abs"]
df_scored["pred_max_y_rel_zf"] = df_scored["pred_max_y_rel"]
df_scored["pred_max_late_zf"] = df_scored["pred_max_late"]

# If Medaka grafted model is available, evaluate (with multi-fold support) and merge strictly on gene_id
if grafted_medaka_model is not None:
    if "grafted_medaka_models" in locals() and len(grafted_medaka_models) > 1:
        print(f"[*] Multi-fold ensemble scoring target species genes across {len(grafted_medaka_models)} Medaka folds...")
        med_folds_scored = []
        for m_fold in grafted_medaka_models:
            df_mf = score_tss_regions(
                grafted_model=m_fold,
                genome_fa_path=TARGET_GENOME_FA,
                gene_df=df_gene_sets,
                flank_bp=25000,
                window_bp=2114,
                stride_bp=250,
                batch_size=64,
                device=device
            )
            med_folds_scored.append(df_mf)
        df_scored_med = med_folds_scored[0].copy()
        for col in ["pred_max_y_abs", "pred_max_y_rel", "pred_max_late"]:
            df_scored_med[col] = np.mean([f[col].values for f in med_folds_scored], axis=0)
    else:
        print(f"[*] Zero-Shot Scoring target species genes using Medaka-trained grafted model...")
        df_scored_med = score_tss_regions(
            grafted_model=grafted_medaka_model,
            genome_fa_path=TARGET_GENOME_FA,
            gene_df=df_gene_sets,
            flank_bp=25000,
            window_bp=2114,
            stride_bp=250,
            batch_size=64,
            device=device
        )
    # Merge on gene_id to prevent any positional row misalignment
    df_scored = df_scored.merge(
        df_scored_med[["gene_id", "pred_max_y_abs", "pred_max_y_rel", "pred_max_late"]],
        on="gene_id",
        how="inner",
        suffixes=("", "_medaka")
    )
    df_scored["pred_max_y_abs"] = 0.5 * (df_scored["pred_max_y_abs_zf"] + df_scored["pred_max_y_abs_medaka"])
    df_scored["pred_max_y_rel"] = 0.5 * (df_scored["pred_max_y_rel_zf"] + df_scored["pred_max_y_rel_medaka"])
    df_scored["pred_max_late"] = 0.5 * (df_scored["pred_max_late_zf"] + df_scored["pred_max_late_medaka"])

# Benchmark evaluations across architectures & metrics
print("=" * 80)
print("[*] Zero-Shot Target Species Cross-Species Transfer Benchmark Report:")

models_eval = [("Zebrafish-only", "_zf")]
if grafted_medaka_model is not None:
    models_eval.append(("Medaka-only", "_medaka"))
    models_eval.append(("Consensus", ""))

for m_label, suffix in models_eval:
    col_abs = f"pred_max_y_abs{suffix}" if suffix else "pred_max_y_abs"
    col_rel = f"pred_max_y_rel{suffix}" if suffix else "pred_max_y_rel"
    col_late = f"pred_max_late{suffix}" if suffix else "pred_max_late"

    print(f"  --- {m_label} Model ---")
    for score_col, score_name in [(col_abs, "y_abs (Pre-MZT Access)"), (col_rel, "y_rel (Timing Ratio)"), (col_late, "late (Late Base Access)")]:
        m_strict = benchmark_transfer_metrics(df_scored, pos_category="strictly_zygotic_minor_wave", ctrl_category="control_non_activated", score_col=score_col)
        m_slam = benchmark_transfer_metrics(df_scored, pos_category="slam_confirmed_minor_wave", ctrl_category="control_non_activated", score_col=score_col)
        m_upreg = benchmark_transfer_metrics(df_scored, pos_category="all_upregulated", ctrl_category="control_non_activated", score_col=score_col)
        print(f"    [{score_name}]:")
        print(f"      1. Strictly Zygotic (Primary):  AUROC = {m_strict['roc_auc']:.3f} | PR-AUC = {m_strict['pr_auc']:.3f} | p = {m_strict['mw_u_pval']:.2e} (n={m_strict['n_pos']})")
        print(f"      2. SLAM-Confirmed (Primary):    AUROC = {m_slam['roc_auc']:.3f} | PR-AUC = {m_slam['pr_auc']:.3f} | p = {m_slam['mw_u_pval']:.2e} (n={m_slam['n_pos']})")
        print(f"      3. All Upregulated (Diagnostic): AUROC = {m_upreg['roc_auc']:.3f} | PR-AUC = {m_upreg['pr_auc']:.3f} | p = {m_upreg['mw_u_pval']:.2e} (n={m_upreg['n_pos']})")

print("=" * 80)

# Zero-Shot Medaka Pre-MZT ATAC Benchmark
medaka_targets_tsv = f"{WORK_DIR}/atac/Oryzias_latipes/Oryzias_latipes_summit_targets.tsv"
if ENABLE_MEDAKA and Path(medaka_targets_tsv).exists() and Path(MEDAKA_GENOME_FA).exists():
    medaka_bench = score_cross_species_medaka_atac(
        grafted_model=grafted_zf_model,
        medaka_targets_tsv=medaka_targets_tsv,
        medaka_fa_path=MEDAKA_GENOME_FA,
        window_bp=2114,
        device=device
    )
    print(f"[+] Zero-Shot Medaka st9 Pre-MZT ATAC: Spearman r = {medaka_bench['medaka_st9_spearman_r']:.3f}, AUROC = {medaka_bench['medaka_st9_auroc']:.3f}")

# Plot Figure 2
metrics_strict = benchmark_transfer_metrics(df_scored, pos_category="strictly_zygotic_minor_wave", ctrl_category="control_non_activated")
metrics_slam = benchmark_transfer_metrics(df_scored, pos_category="slam_confirmed_minor_wave", ctrl_category="control_non_activated")
metrics_upreg = benchmark_transfer_metrics(df_scored, pos_category="all_upregulated", ctrl_category="control_non_activated")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
for cat, label, color, ls in [
    ("strictly_zygotic_minor_wave", f"Strictly Zygotic (AUC={metrics_strict['roc_auc']:.2f})", "#d7191c", "-"),
    ("slam_confirmed_minor_wave", f"SLAM-Confirmed (AUC={metrics_slam['roc_auc']:.2f})", "#2b83ba", "-"),
    ("all_upregulated", f"All Upregulated (Diagnostic, AUC={metrics_upreg['roc_auc']:.2f})", "#fdae61", "--")
]:
    if cat == "slam_confirmed_minor_wave":
        pos_mask = (df_scored.get("is_slam_confirmed_minor_wave", df_scored.get("is_slam_confirmed", False)) == True) | (df_scored["category"] == "slam_confirmed_minor_wave")
    elif cat == "all_upregulated":
        pos_mask = (df_scored.get("is_all_upregulated", False) == True) | (df_scored["category"].isin(["all_upregulated", "strictly_zygotic_minor_wave"]))
    elif cat == "strictly_zygotic_minor_wave":
        pos_mask = (df_scored.get("is_strictly_zygotic_minor_wave", False) == True) | (df_scored["category"] == "strictly_zygotic_minor_wave")
    else:
        pos_mask = df_scored["category"] == cat
    pos_s = df_scored.loc[pos_mask, "pred_max_y_abs"].dropna().values
    ctrl_s = df_scored.loc[df_scored["category"] == "control_non_activated", "pred_max_y_abs"].dropna().values
    if len(pos_s) > 0 and len(ctrl_s) > 0:
        y_true = np.concatenate([np.ones_like(pos_s), np.zeros_like(ctrl_s)])
        y_score = np.concatenate([pos_s, ctrl_s])
        fpr, tpr, _ = roc_curve(y_true, y_score)
        ax.plot(fpr, tpr, label=label, color=color, lw=2.2, linestyle=ls)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title("Zero-Shot Cross-Species ROC Curves", fontweight="bold")
ax.legend(frameon=True, fontsize=9, loc="lower right")

ax = axes[1]
for cat, label, color, ls in [
    ("strictly_zygotic_minor_wave", f"Strictly Zygotic (PR={metrics_strict['pr_auc']:.2f})", "#d7191c", "-"),
    ("slam_confirmed_minor_wave", f"SLAM-Confirmed (PR={metrics_slam['pr_auc']:.2f})", "#2b83ba", "-"),
    ("all_upregulated", f"All Upregulated (Diagnostic, PR={metrics_upreg['pr_auc']:.2f})", "#fdae61", "--")
]:
    if cat == "slam_confirmed_minor_wave":
        pos_mask = (df_scored.get("is_slam_confirmed_minor_wave", df_scored.get("is_slam_confirmed", False)) == True) | (df_scored["category"] == "slam_confirmed_minor_wave")
    elif cat == "all_upregulated":
        pos_mask = (df_scored.get("is_all_upregulated", False) == True) | (df_scored["category"].isin(["all_upregulated", "strictly_zygotic_minor_wave"]))
    elif cat == "strictly_zygotic_minor_wave":
        pos_mask = (df_scored.get("is_strictly_zygotic_minor_wave", False) == True) | (df_scored["category"] == "strictly_zygotic_minor_wave")
    else:
        pos_mask = df_scored["category"] == cat
    pos_s = df_scored.loc[pos_mask, "pred_max_y_abs"].dropna().values
    ctrl_s = df_scored.loc[df_scored["category"] == "control_non_activated", "pred_max_y_abs"].dropna().values
    if len(pos_s) > 0 and len(ctrl_s) > 0:
        y_true = np.concatenate([np.ones_like(pos_s), np.zeros_like(ctrl_s)])
        y_score = np.concatenate([pos_s, ctrl_s])
        prec, rec, _ = precision_recall_curve(y_true, y_score)
        ax.plot(rec, prec, label=label, color=color, lw=2.2, linestyle=ls)
ax.set_xlabel("Recall", fontsize=11)
ax.set_ylabel("Precision", fontsize=11)
ax.set_title("Zero-Shot Precision-Recall Curves", fontweight="bold")
ax.legend(frameon=True, fontsize=9, loc="lower left")
plt.tight_layout()
fig2_path = res_dir / "fig2_zeroshot_benchmark.png"
plt.savefig(fig2_path, dpi=300)
plt.show()
print(f"[+] Figure 2 exported to {fig2_path}")

# Persist scored gene metrics table
df_scored.to_csv(res_dir / "target_genes_scored_metrics.tsv", sep="\t", index=False)
print(f"[+] Persisted target genes scored metrics to {res_dir / 'target_genes_scored_metrics.tsv'}")

---
## Block 5: Unbiased Sliding-Window Scan of `TARGET_LOCUS_NAME`

We tile the target locus across species to discover putative gained enhancers driving pre-MZT activation:
- Dynamically discovers coordinates for `TARGET_GENES` (`notch2`, `slc35a3a`) directly from `TARGET_GTF`.
- Pads $\pm \text{LOCUS\_FLANK\_BP}$ (75 kb) to capture distal enhancers.
- Exports candidate enhancer BED and bedGraph tracks to `WORK_DIR/results/{TARGET_LOCUS_NAME}/`.

In [ ]:
# Unbiased Sliding-Window Scanning across Target Locus & Cross-Species Loci
from scripts.infer_and_attribute import scan_locus_sliding_window
import matplotlib.pyplot as plt
import pyBigWig
import gzip

gene_sets_tsv = res_dir / f"{TARGET_LOCUS_NAME}_gene_sets.tsv"
df_counts = pd.read_csv(gene_sets_tsv, sep="\t")

# Store (gname, chrom, st, en) tuples so labels never misalign
tg_coords = []
for tg in TARGET_GENES:
    tg_rows = df_counts[df_counts["gene_name"].str.lower() == tg.lower()]
    if not tg_rows.empty:
        r = tg_rows.iloc[0]
        tg_coords.append((r["gene_name"], str(r["chrom"]), int(r["start"]), int(r["end"])))

assert tg_coords, f"None of the target genes {TARGET_GENES} found in GTF!"
locus_chr = tg_coords[0][1]
locus_start = max(0, min(c[2] for c in tg_coords) - LOCUS_FLANK_BP)
locus_end = max(c[3] for c in tg_coords) + LOCUS_FLANK_BP
locus_len = locus_end - locus_start

print(f"[+] Scanning locus {TARGET_LOCUS_NAME}: {locus_chr}:{locus_start:,}-{locus_end:,} ({locus_len:,} bp)")
window_bp = 2114
stride_bp = 250
device = "cuda" if torch.cuda.is_available() else "cpu"

if "grafted_zf_models" in locals() and len(grafted_zf_models) > 1:
    print(f"[*] Multi-fold ensemble scanning target locus across {len(grafted_zf_models)} Zebrafish folds...")
    zf_abs_list, zf_rel_list, zf_late_list = [], [], []
    for m_fold in grafted_zf_models:
        pos_f, abs_f, rel_f, late_f = scan_locus_sliding_window(
            grafted_model=m_fold,
            genome_fa_path=TARGET_GENOME_FA,
            chrom=locus_chr,
            start_bp=locus_start,
            end_bp=locus_end,
            window_bp=window_bp,
            stride_bp=stride_bp,
            batch_size=64,
            device=device,
            return_late=True
        )
        zf_abs_list.append(abs_f)
        zf_rel_list.append(rel_f)
        zf_late_list.append(late_f)
    positions = pos_f
    target_pred_abs_zf = np.mean(zf_abs_list, axis=0)
    target_pred_rel_zf = np.mean(zf_rel_list, axis=0)
    target_pred_late_zf = np.mean(zf_late_list, axis=0)
else:
    positions, target_pred_abs_zf, target_pred_rel_zf, target_pred_late_zf = scan_locus_sliding_window(
        grafted_model=grafted_zf_model,
        genome_fa_path=TARGET_GENOME_FA,
        chrom=locus_chr,
        start_bp=locus_start,
        end_bp=locus_end,
        window_bp=window_bp,
        stride_bp=stride_bp,
        batch_size=64,
        device=device,
        return_late=True
    )

target_pred_abs = target_pred_abs_zf
target_pred_rel = target_pred_rel_zf
target_pred_late = target_pred_late_zf

# Multi-fold ensembling for Medaka models whenever len(grafted_medaka_models) > 1
if grafted_medaka_model is not None:
    if "grafted_medaka_models" in locals() and len(grafted_medaka_models) > 1:
        print(f"[*] Multi-fold ensemble scanning target locus across {len(grafted_medaka_models)} Medaka folds...")
        med_abs_list, med_rel_list, med_late_list = [], [], []
        for m_fold in grafted_medaka_models:
            _, m_abs, m_rel, m_late = scan_locus_sliding_window(
                grafted_model=m_fold,
                genome_fa_path=TARGET_GENOME_FA,
                chrom=locus_chr,
                start_bp=locus_start,
                end_bp=locus_end,
                window_bp=window_bp,
                stride_bp=stride_bp,
                batch_size=64,
                device=device,
                return_late=True
            )
            med_abs_list.append(m_abs)
            med_rel_list.append(m_rel)
            med_late_list.append(m_late)
        med_abs = np.mean(med_abs_list, axis=0)
        med_rel = np.mean(med_rel_list, axis=0)
        med_late = np.mean(med_late_list, axis=0)
    else:
        _, med_abs, med_rel, med_late = scan_locus_sliding_window(
            grafted_model=grafted_medaka_model,
            genome_fa_path=TARGET_GENOME_FA,
            chrom=locus_chr,
            start_bp=locus_start,
            end_bp=locus_end,
            window_bp=window_bp,
            stride_bp=stride_bp,
            batch_size=64,
            device=device,
            return_late=True
        )
    target_pred_abs = 0.5 * (target_pred_abs_zf + med_abs)
    target_pred_rel = 0.5 * (target_pred_rel_zf + med_rel)
    target_pred_late = 0.5 * (target_pred_late_zf + med_late)

enhancer_threshold = np.percentile(target_pred_abs, 95)
is_enhancer_raw = target_pred_abs > enhancer_threshold

# Non-maximum suppression with min_distance = 1500 bp
cand_indices = np.where(is_enhancer_raw)[0]
cand_sorted = cand_indices[np.argsort(-target_pred_abs[cand_indices])]
kept_centers = []
is_enhancer = np.zeros(len(positions), dtype=bool)

for idx in cand_sorted:
    pos = positions[idx]
    if not any(abs(pos - kc) < 1500 for kc in kept_centers):
        kept_centers.append(pos)
        is_enhancer[idx] = True

print(f"[+] Identified {len(kept_centers)} non-redundant candidate enhancer summits after NMS (min_distance=1500 bp).")

# Export BED, TSV, and IGV-ready bedGraph tracks
bed_path = res_dir / f"{TARGET_LOCUS_NAME}_candidate_enhancers.bed"
tsv_path = res_dir / f"{TARGET_LOCUS_NAME}_candidate_enhancers.tsv"
bg_abs_path = res_dir / f"{TARGET_LOCUS_NAME}_pred_y_abs.bedgraph"
bg_rel_path = res_dir / f"{TARGET_LOCUS_NAME}_pred_y_rel.bedgraph"
bg_late_path = res_dir / f"{TARGET_LOCUS_NAME}_pred_late.bedgraph"

half_stride = stride_bp // 2
records = []
with open(bed_path, "w") as f_bed, open(bg_abs_path, "w") as f_bg_a, open(bg_rel_path, "w") as f_bg_r, open(bg_late_path, "w") as f_bg_l:
    for idx, (pos, is_e) in enumerate(zip(positions, is_enhancer)):
        st = max(0, pos - window_bp // 2)
        en = st + window_bp
        # Non-overlapping step interval centered at pos for IGV and bedGraphToBigWig
        step_st = max(locus_start, pos - half_stride)
        step_en = max(step_st + 1, min(locus_end, pos + half_stride))
        score_a = float(target_pred_abs[idx])
        score_r = float(target_pred_rel[idx])
        score_l = float(target_pred_late[idx])
        records.append({
            "chrom": locus_chr,
            "start": st,
            "end": en,
            "center": pos,
            "pred_abs": round(score_a, 4),
            "pred_rel": round(score_r, 4),
            "pred_late": round(score_l, 4),
            "is_enhancer": bool(is_e),
            "is_candidate": bool(is_e)
        })
        if is_e:
            f_bed.write(f"{locus_chr}\t{st}\t{en}\t{TARGET_LOCUS_NAME}_peak_{idx}\t{int(score_a * 100)}\t.\t{pos}\t{pos}\n")
        f_bg_a.write(f"{locus_chr}\t{step_st}\t{step_en}\t{score_a:.4f}\n")
        f_bg_r.write(f"{locus_chr}\t{step_st}\t{step_en}\t{score_r:.4f}\n")
        f_bg_l.write(f"{locus_chr}\t{step_st}\t{step_en}\t{score_l:.4f}\n")

df_enh = pd.DataFrame(records)
df_enh.to_csv(tsv_path, sep="\t", index=False)
print(f"[+] Candidate enhancers BED written to {bed_path}")
print(f"[+] Candidate enhancers TSV written to {tsv_path}")
print(f"[+] IGV-ready bedGraph tracks: {bg_abs_path.name}, {bg_rel_path.name}")

# Cross-Species Scanning: Zebrafish Orthologous Locus
zf_locus_info = None
zf_pred_abs = None
try:
    zf_genes_dict = {}
    with (gzip.open(ZEBRAFISH_GTF, "rt") if str(ZEBRAFISH_GTF).endswith(".gz") else open(ZEBRAFISH_GTF, "rt")) as f:
        for line in f:
            if line.startswith("#"):
                continue
            p = line.strip().split("\t")
            if len(p) >= 9 and p[2] == "gene":
                for tg in TARGET_GENES:
                    tg_l = tg.lower()
                    if f'gene_name "{tg_l}"' in p[8].lower() or f'"{tg_l}"' in p[8].lower():
                        zf_genes_dict[tg_l] = (p[0], int(p[3]), int(p[4]))
    if zf_genes_dict:
        sample_gene = list(zf_genes_dict.values())[0]
        zf_chr = sample_gene[0]
        zf_st = max(0, min(v[1] for v in zf_genes_dict.values()) - LOCUS_FLANK_BP)
        zf_en = max(v[2] for v in zf_genes_dict.values()) + LOCUS_FLANK_BP
        print(f"[+] Scanning orthologous Zebrafish locus: {zf_chr}:{zf_st:,}-{zf_en:,}")
        zf_positions, zf_pred_abs, zf_pred_rel = scan_locus_sliding_window(
            grafted_model=grafted_zf_model,
            genome_fa_path=ZEBRAFISH_GENOME_FA,
            chrom=zf_chr,
            start_bp=zf_st,
            end_bp=zf_en,
            window_bp=window_bp,
            stride_bp=stride_bp,
            batch_size=64,
            device=device
        )
        zf_locus_info = (zf_chr, zf_st, zf_en, zf_positions, zf_pred_abs)
except Exception as e:
    print(f"[-] Notice scanning Zebrafish orthologous locus: {e}")

# Cross-Species Scanning: Medaka Orthologous Locus (when ENABLE_MEDAKA)
med_locus_info = None
if ENABLE_MEDAKA and Path(MEDAKA_GTF).exists() and Path(MEDAKA_GENOME_FA).exists():
    try:
        med_genes_dict = {}
        with (gzip.open(MEDAKA_GTF, "rt") if str(MEDAKA_GTF).endswith(".gz") else open(MEDAKA_GTF, "rt")) as f:
            for line in f:
                if line.startswith("#"):
                    continue
                p = line.strip().split("\t")
                if len(p) >= 9 and p[2] == "gene":
                    for tg in TARGET_GENES:
                        tg_l = tg.lower()
                        # Match exact first, then fallback to stripping a single trailing a/b
                        if f'gene_name "{tg_l}"' in p[8].lower() or f'"{tg_l}"' in p[8].lower():
                            med_genes_dict[tg_l] = (p[0], int(p[3]), int(p[4]))
                        elif len(tg_l) > 3 and tg_l[-1] in "ab":
                            tg_stem = tg_l[:-1]
                            if f'gene_name "{tg_stem}"' in p[8].lower() or f'"{tg_stem}"' in p[8].lower():
                                med_genes_dict[tg_stem] = (p[0], int(p[3]), int(p[4]))
        if med_genes_dict:
            sample_m = list(med_genes_dict.values())[0]
            m_chr = sample_m[0]
            m_st = max(0, min(v[1] for v in med_genes_dict.values()) - LOCUS_FLANK_BP)
            m_en = max(v[2] for v in med_genes_dict.values()) + LOCUS_FLANK_BP
            print(f"[+] Scanning orthologous Medaka locus: {m_chr}:{m_st:,}-{m_en:,}")
            if "grafted_medaka_models" in locals() and len(grafted_medaka_models) > 1:
                print(f"[*] Multi-fold ensemble scanning orthologous Medaka locus across {len(grafted_medaka_models)} Medaka folds...")
                m_abs_list, m_rel_list = [], []
                for m_fold in grafted_medaka_models:
                    m_positions, m_abs, m_rel = scan_locus_sliding_window(
                        grafted_model=m_fold,
                        genome_fa_path=MEDAKA_GENOME_FA,
                        chrom=m_chr,
                        start_bp=m_st,
                        end_bp=m_en,
                        window_bp=window_bp,
                        stride_bp=stride_bp,
                        batch_size=64,
                        device=device
                    )
                    m_abs_list.append(m_abs)
                    m_rel_list.append(m_rel)
                m_pred_abs = np.mean(m_abs_list, axis=0)
            else:
                med_model_to_use = grafted_medaka_model if grafted_medaka_model is not None else grafted_zf_model
                m_positions, m_pred_abs, m_pred_rel = scan_locus_sliding_window(
                    grafted_model=med_model_to_use,
                    genome_fa_path=MEDAKA_GENOME_FA,
                    chrom=m_chr,
                    start_bp=m_st,
                    end_bp=m_en,
                    window_bp=window_bp,
                    stride_bp=stride_bp,
                    batch_size=64,
                    device=device
                )
            med_locus_info = (m_chr, m_st, m_en, m_positions, m_pred_abs)
    except Exception as e:
        print(f"[-] Notice scanning Medaka orthologous locus: {e}")

# Synchronized Cross-Species Visualization Panel
panel_count = 3
if zf_locus_info is not None:
    panel_count += 1
if med_locus_info is not None:
    panel_count += 1
fig, axes = plt.subplots(panel_count, 1, figsize=(14, 3 * panel_count), sharex=False)

# Panel 1: Target Species Pre-MZT Accessibility
axes[0].plot(positions / 1e6, target_pred_abs, color="#d7191c", lw=1.8, label=f"{TARGET_SPECIES_NAME} Pre-MZT Accessibility (y_abs)")
axes[0].plot(positions / 1e6, target_pred_late, color="#fd8d3c", lw=1.2, linestyle="--", alpha=0.8, label=f"{TARGET_SPECIES_NAME} Late Accessibility (pred_late)")
axes[0].axhline(enhancer_threshold, color="black", linestyle="--", label="95th %ile Threshold")
axes[0].set_ylabel("Predicted y_abs", fontsize=10)
axes[0].set_title(f"Target Species Locus ({TARGET_SPECIES_NAME}: {locus_chr}:{locus_start:,}-{locus_end:,})", fontweight="bold")
axes[0].legend(loc="upper right", fontsize=9)

# Panel 2: Target Species Timing Ratio
axes[1].plot(positions / 1e6, target_pred_rel, color="#2b83ba", lw=1.8, label="Timing Ratio (y_rel)")
axes[1].axhline(0, color="gray", linestyle=":")
axes[1].set_ylabel("Predicted y_rel", fontsize=10)
axes[1].legend(loc="upper right", fontsize=9)

# Panel 3: Target Gene Models
ax_genes = axes[2]
ax_genes.set_ylim(-0.5, 1.5)
for gname, chrom, st, en in tg_coords:
    ax_genes.plot([st / 1e6, en / 1e6], [0.5, 0.5], lw=6, color="#252525")
    ax_genes.annotate(gname, ((st + en)/2e6, 0.8), ha="center", fontsize=11, fontweight="bold")
ax_genes.set_yticks([])
ax_genes.set_ylabel("Gene Models", fontsize=10)
ax_genes.set_xlabel(f"Genomic Coordinate on Chr {locus_chr} (Mb)", fontsize=11)

# Cross-Species Orthologous Loci & Observed BigWig Overlay Panels (with chromosome end clamping)
curr_ax_idx = 3
if zf_locus_info is not None:
    z_chr, z_st, z_en, z_pos, z_abs = zf_locus_info
    ax_zf = axes[curr_ax_idx]
    curr_ax_idx += 1
    ax_zf.plot(z_pos / 1e6, z_abs, color="#756bb1", lw=1.8, label="Zebrafish Ortholog Predicted y_abs")
    zf_early_bw = Path(WORK_DIR) / "atac" / "Danio_rerio" / "Danio_rerio_early_cpm.bw"
    if zf_early_bw.exists():
        try:
            with pyBigWig.open(str(zf_early_bw)) as bw:
                chrom_lens = bw.chroms()
                z_max = chrom_lens.get(z_chr, z_en)
                z_en_clamped = min(z_en, z_max)
                z_st_clamped = min(z_st, max(0, z_en_clamped - 1))
                vals = np.nan_to_num(bw.values(z_chr, z_st_clamped, z_en_clamped))
            bw_x = np.linspace(z_st_clamped / 1e6, z_en_clamped / 1e6, len(vals))
            ax_zf.plot(bw_x, np.log2(vals + 1.0), color="#2ca25f", alpha=0.6, lw=1.0, label="Observed 256-cell ATAC CPM")
        except Exception as e:
            print(f"[-] Notice reading Zebrafish BigWig: {e}")
    ax_zf.set_ylabel("y_abs / ATAC", fontsize=10)
    ax_zf.set_xlabel(f"Zebrafish Chr {z_chr} (Mb)", fontsize=11)
    ax_zf.set_title(f"Orthologous Zebrafish Locus ({z_chr}:{z_st:,}-{z_en:,})", fontweight="bold")
    ax_zf.legend(loc="upper right", fontsize=9)

if med_locus_info is not None:
    m_chr, m_st, m_en, m_pos, m_abs = med_locus_info
    ax_med = axes[curr_ax_idx]
    curr_ax_idx += 1
    ax_med.plot(m_pos / 1e6, m_abs, color="#31a354", lw=1.8, label="Medaka Ortholog Predicted y_abs")
    med_early_bw = Path(WORK_DIR) / "atac" / "Oryzias_latipes" / "Oryzias_latipes_early_cpm.bw"
    if med_early_bw.exists():
        try:
            with pyBigWig.open(str(med_early_bw)) as bw:
                chrom_lens_m = bw.chroms()
                m_max = chrom_lens_m.get(m_chr, m_en)
                m_en_clamped = min(m_en, m_max)
                m_st_clamped = min(m_st, max(0, m_en_clamped - 1))
                vals_m = np.nan_to_num(bw.values(m_chr, m_st_clamped, m_en_clamped))
            bw_m_x = np.linspace(m_st_clamped / 1e6, m_en_clamped / 1e6, len(vals_m))
            ax_med.plot(bw_m_x, np.log2(vals_m + 1.0), color="#fd8d3c", alpha=0.6, lw=1.0, label="Observed st9 ATAC CPM")
        except Exception as e:
            print(f"[-] Notice reading Medaka BigWig: {e}")
    ax_med.set_ylabel("y_abs / ATAC", fontsize=10)
    ax_med.set_xlabel(f"Medaka Chr {m_chr} (Mb)", fontsize=11)
    ax_med.set_title(f"Orthologous Medaka Locus ({m_chr}:{m_st:,}-{m_en:,})", fontweight="bold")
    ax_med.legend(loc="upper right", fontsize=9)

plt.tight_layout()
fig3_path = res_dir / "fig3_locus_scan_profile.png"
plt.savefig(fig3_path, dpi=300)
plt.savefig(res_dir / "fig3_locus_scan_profile.svg")
plt.show()
print(f"[+] Figure 3 exported to {fig3_path}")

---
## Block 6: Base-Resolution Attribution, Teleost-Normalized Pre-MZT TF Filtering & Evolutionary Swap

We dissect the causal sequence grammar of top candidate enhancers driving early activation:
1. **Gradient Saliency & In Silico Mutagenesis (ISM):** Compute base-resolution contribution and sensitivity logos.
2. **Pre-MZT TF Matching via TOMTOM:** Query causal seqlets against both:
   - Filtered `premzt_tfs.meme` with teleost paralog normalization (`POU5F3` $\rightarrow$ `POU5F1`, `MXTX2` $\rightarrow$ `MIXL1`, `VOX` $\rightarrow$ `VENTX`).
   - Full JASPAR database for unbiased screening.
3. **Reciprocal In Silico Evolutionary Swap:** Exchange top causal motifs between target species and Zebrafish orthologs, evaluating $\Delta$ predicted pre-MZT accessibility.

In [ ]:
# Base-Resolution Attribution, In Silico Mutagenesis (ISM) & Evolutionary Swap
from scripts.infer_and_attribute import (
    one_hot_encode_sequence,
    compute_input_gradient_attribution,
    compute_in_silico_mutagenesis,
    compute_wt_projected_ism_matrix,
    seqlet_to_meme,
    run_tomtom,
    annotate_motif_hits,
    filter_jaspar_meme,
    perform_evolutionary_swap
)
import logomaker
from pyfaidx import Fasta
import matplotlib.pyplot as plt
import gzip

# 1. Select Top K Non-Redundant Candidate Enhancers (NMS peaks strictly filtered)
cands_pool = df_enh[df_enh.get("is_candidate", df_enh.get("is_enhancer", True))]
if cands_pool.empty:
    cands_pool = df_enh
top_k = min(5, len(cands_pool))
candidates_top = cands_pool.sort_values(by="pred_abs", ascending=False).head(top_k)
top_enhancer = candidates_top.iloc[0]
best_pos = int(top_enhancer["center"])
print(f"[+] Evaluating Top {top_k} Non-Redundant Candidate Enhancers:")
for i, (_, cand) in enumerate(candidates_top.iterrows(), 1):
    print(f"    Enhancer {i}: {cand['chrom']}:{int(cand['start']):,}-{int(cand['end']):,} | pred_abs = {cand['pred_abs']:.3f} | pred_rel = {cand['pred_rel']:.3f}")

# Extract top enhancer sequence
target_genome = Fasta(TARGET_GENOME_FA, as_raw=True)
enh_seq = str(target_genome[locus_chr][best_pos - window_bp // 2 : best_pos + window_bp // 2]).upper()
one_hot_enh = one_hot_encode_sequence(enh_seq)

# Loop attribution profiling across top candidate peaks
center_offset = window_bp // 2
search_flank = min(400, center_offset)
all_cands_analysis = []
for cand_idx, (_, cand_row) in enumerate(candidates_top.iterrows()):
    c_pos = int(cand_row["center"])
    c_seq = str(target_genome[locus_chr][c_pos - window_bp // 2 : c_pos + window_bp // 2]).upper()
    c_onehot = one_hot_encode_sequence(c_seq)
    c_attr = compute_input_gradient_attribution(grafted_zf_model, c_onehot, target="y_abs", device=device)
    c_scores = np.sum(np.abs(c_attr), axis=0)
    c_peak = (center_offset - search_flank) + int(np.argmax(c_scores[center_offset - search_flank : center_offset + search_flank]))
    all_cands_analysis.append({
        "enhancer_rank": cand_idx + 1,
        "chrom": cand_row["chrom"],
        "start": int(cand_row["start"]),
        "end": int(cand_row["end"]),
        "center": c_pos,
        "gradient_peak_offset": c_peak,
        "pred_abs": float(cand_row["pred_abs"]),
        "pred_rel": float(cand_row["pred_rel"]),
        "pred_late": float(cand_row.get("pred_late", cand_row["pred_abs"]))
    })

print(f"[+] Multi-candidate attribution profiling completed for {len(all_cands_analysis)} non-redundant peaks.")

# Export top candidates attribution analysis
df_cands_attr = pd.DataFrame(all_cands_analysis)
cands_tsv_path = res_dir / f"{TARGET_LOCUS_NAME}_top_candidates_attribution.tsv"
df_cands_attr.to_csv(cands_tsv_path, sep="\t", index=False)
print(f"[+] Saved top candidates attribution analysis to {cands_tsv_path}")

# 2. Input x Gradient & In Silico Mutagenesis (ISM) on Top Candidate
grad_attr = compute_input_gradient_attribution(grafted_zf_model, one_hot_enh, target="y_abs", device=device)
grad_pos_scores = np.sum(np.abs(grad_attr), axis=0)
grad_peak = (center_offset - search_flank) + int(np.argmax(grad_pos_scores[center_offset - search_flank : center_offset + search_flank]))

ism_region_start = max(0, min(window_bp - 60, grad_peak - 30))
ism_region_len = 60
print(f"[*] Gradient peak detected at position {grad_peak} (centering 60 bp ISM at {ism_region_start}-{ism_region_start+60})...")
ism_mat, wt_importance = compute_in_silico_mutagenesis(
    grafted_model=grafted_zf_model,
    wt_one_hot=one_hot_enh,
    region_start=ism_region_start,
    region_len=ism_region_len,
    target="y_abs",
    batch_size=64,
    device=device
)

# WT-projected importance matrix so activating WT bases render positively in logomaker logos
wt_sub_onehot = one_hot_enh[:, ism_region_start : ism_region_start + ism_region_len]
wt_logo_mat = compute_wt_projected_ism_matrix(ism_mat, wt_sub_onehot, wt_importance)
df_logo = pd.DataFrame(wt_logo_mat.T, columns=["A", "C", "G", "T"])
df_ism = pd.DataFrame(ism_mat.T, columns=["A", "C", "G", "T"])

# Evaluate timing impact (y_rel)
ism_mat_rel, wt_importance_rel = compute_in_silico_mutagenesis(
    grafted_model=grafted_zf_model,
    wt_one_hot=one_hot_enh,
    region_start=ism_region_start,
    region_len=ism_region_len,
    target="y_rel",
    batch_size=64,
    device=device
)
print(f"[+] Timing impact (y_rel): mean ablation delta = {float(np.mean(ism_mat_rel)):+.3f}")

# Paired Medaka model attribution/logo when available
df_logo_med = None
if grafted_medaka_model is not None:
    try:
        ism_mat_med, wt_imp_med = compute_in_silico_mutagenesis(
            grafted_model=grafted_medaka_model,
            wt_one_hot=one_hot_enh,
            region_start=ism_region_start,
            region_len=ism_region_len,
            target="y_abs",
            batch_size=64,
            device=device
        )
        wt_logo_med = compute_wt_projected_ism_matrix(ism_mat_med, wt_sub_onehot, wt_imp_med)
        df_logo_med = pd.DataFrame(wt_logo_med.T, columns=["A", "C", "G", "T"])
        print("[+] Computed paired Medaka model WT-projected ISM matrix.")
    except Exception as e:
        print(f"[-] Notice computing paired Medaka attribution: {e}")

motif_len = 10
motif_scores = [wt_importance[i:i + motif_len].sum() for i in range(len(wt_importance) - motif_len)]
best_offset = int(np.argmax(motif_scores))
causal_sub_ism = ism_mat[:, best_offset:best_offset + motif_len]
causal_seq = enh_seq[ism_region_start + best_offset : ism_region_start + best_offset + motif_len]
print(f"[+] Core Causal Motif identified at relative position {best_offset}: '{causal_seq}'")

# 3. Export Motif PPM directly from signed ISM matrix via softmax
query_meme = res_dir / "candidate_enhancer_causal_motif.meme"
seqlet_to_meme(ppm_or_pwm=causal_sub_ism, motif_id="CAUSAL_PREMZT_1", out_meme_path=query_meme, beta=10.0)

# Build maternal RNA-seq CPM dictionary from Block 1 df_counts
rnaseq_cpm_dict = dict(zip(df_counts["gene_name"].str.lower(), df_counts["cpm_early"]))

# 4. TOMTOM Motif Matching against Pre-MZT TFs AND Full JASPAR
df_tomtom_premzt = run_tomtom(query_meme_path=query_meme, target_meme_path=filtered_meme_path, thresh=0.5)
df_tomtom_premzt = annotate_motif_hits(
    df_tomtom_premzt,
    tf_list_path=PREMZT_TF_LIST_PATH,
    rnaseq_cpm_dict=rnaseq_cpm_dict,
    target_meme_path=filtered_meme_path
)

df_tomtom_jaspar = run_tomtom(query_meme_path=query_meme, target_meme_path=JASPAR_MEME_PATH, thresh=0.5)
df_tomtom_jaspar = annotate_motif_hits(
    df_tomtom_jaspar,
    tf_list_path=PREMZT_TF_LIST_PATH,
    rnaseq_cpm_dict=rnaseq_cpm_dict,
    target_meme_path=JASPAR_MEME_PATH
)

print("=" * 80)
print(f"[*] TOMTOM Motif Matching Results (Pre-MZT TF Database):")
if not df_tomtom_premzt.empty:
    disp_cols = [c for c in ["Query_ID", "Target_ID", "p-value", "q-value", "is_premzt_tf", "maternal_cpm"] if c in df_tomtom_premzt.columns]
    print(df_tomtom_premzt[disp_cols].head(5).to_string(index=False))
else:
    print("    No significant hits in curated pre-MZT TF subset.")

df_tomtom_jaspar.to_csv(res_dir / "tomtom_jaspar.tsv", sep="\t", index=False)
print(f"[+] Saved JASPAR TOMTOM results to {res_dir / 'tomtom_jaspar.tsv'}")
print(f"[*] TOMTOM Motif Matching Results (Full JASPAR Vertebrates):")
if not df_tomtom_jaspar.empty:
    disp_cols = [c for c in ["Query_ID", "Target_ID", "p-value", "q-value", "is_premzt_tf", "maternal_cpm"] if c in df_tomtom_jaspar.columns]
    print(df_tomtom_jaspar[disp_cols].head(5).to_string(index=False))
print("=" * 80)

# 5. Dynamic Orthologous Locus Extraction from ZEBRAFISH_GTF (Fix hardcoded chr3 bug & remove random fallback)
zf_genome = Fasta(ZEBRAFISH_GENOME_FA, as_raw=True)
zf_locus_seq = ""
zf_chr = None
try:
    with (gzip.open(ZEBRAFISH_GTF, "rt") if str(ZEBRAFISH_GTF).endswith(".gz") else open(ZEBRAFISH_GTF, "rt")) as f:
        zf_t_coords = []
        for line in f:
            if line.startswith("#"):
                continue
            p = line.strip().split("\t")
            if len(p) >= 9 and p[2] == "gene":
                for tg in TARGET_GENES:
                    tg_l = tg.lower()
                    if f'gene_name "{tg_l}"' in p[8].lower() or f'"{tg_l}"' in p[8].lower():
                        zf_t_coords.append((p[0], int(p[3]), int(p[4])))
    if zf_t_coords:
        zf_chr = zf_t_coords[0][0]
        zf_min = max(0, min(c[1] for c in zf_t_coords) - LOCUS_FLANK_BP)
        zf_max = max(c[2] for c in zf_t_coords) + LOCUS_FLANK_BP
        zf_locus_seq = str(zf_genome[zf_chr][zf_min:zf_max]).upper()
        print(f"[+] Dynamically extracted Zebrafish orthologous locus: {zf_chr}:{zf_min:,}-{zf_max:,} ({len(zf_locus_seq):,} bp)")
except Exception as e:
    print(f"[-] Notice dynamically extracting Zebrafish locus: {e}")

if not zf_locus_seq:
    print(f"[-] Warning: Zebrafish orthologous coordinates not found for {TARGET_GENES}. Skipping evolutionary swap (avoiding random DNA fallback).")
    swap_results = {
        "classification": "skipped_no_ortholog",
        "target_ablation_delta": 0.0,
        "target_wt": 0.0,
        "target_ablation_score": 0.0,
        "target_swap_score": None,
        "target_swap_delta": None,
        "ortholog_aligned_motif": None
    }
else:
    # 6. Evolutionary Alignment & Swap (Forward + Reverse Strands)
    swap_results = perform_evolutionary_swap(
        model=grafted_zf_model,
        target_species_seq=enh_seq,
        ortholog_species_seq=zf_locus_seq,
        motif_start=ism_region_start + best_offset,
        motif_len=motif_len,
        device=device
    )

print("=" * 80)
pd.DataFrame([swap_results]).to_csv(res_dir / "evolutionary_swap_summary.tsv", sep="\t", index=False)
print(f"[+] Saved evolutionary swap summary to {res_dir / 'evolutionary_swap_summary.tsv'}")
print("=" * 80)
print(f"[*] Evolutionary Motif Perturbation & Swap Results:")
print(f"    Classification:        {swap_results.get('classification')}")
print(f"    Target WT:             {swap_results.get('target_wt', 0.0):.3f}")
print(f"    In Silico Ablation:    {swap_results.get('target_ablation_score', 0.0):.3f} (Δ = {swap_results.get('target_ablation_delta', 0.0):+.3f})")
if swap_results.get("ortholog_swap_delta") is not None:
    print(f"    Ortholog Swap Δ:       {swap_results.get('ortholog_swap_delta'):+.3f}")
print("=" * 80)

# Plot Figure 4 (with paired Medaka model logo comparison when available)
if df_logo_med is not None:
    fig = plt.figure(figsize=(16, 8))
    gs = fig.add_gridspec(3, 2, width_ratios=[2.5, 1])

    ax_logo1 = fig.add_subplot(gs[0, 0])
    sub_logo = df_logo.iloc[max(0, best_offset - 4) : min(len(df_logo), best_offset + motif_len + 4)].reset_index(drop=True)
    logomaker.Logo(sub_logo, ax=ax_logo1)
    ax_logo1.set_title(f"Zebrafish-Model Causal Motif: '{causal_seq}' (WT-Projected ISM)", fontweight="bold")
    ax_logo1.set_ylabel("WT Importance")

    ax_logo_med = fig.add_subplot(gs[1, 0])
    sub_logo_med = df_logo_med.iloc[max(0, best_offset - 4) : min(len(df_logo_med), best_offset + motif_len + 4)].reset_index(drop=True)
    logomaker.Logo(sub_logo_med, ax=ax_logo_med)
    ax_logo_med.set_title(f"Medaka-Model Paired Causal Motif (WT-Projected ISM)", fontweight="bold")
    ax_logo_med.set_ylabel("WT Importance")

    ax_logo2 = fig.add_subplot(gs[2, 0])
    logomaker.Logo(df_logo, ax=ax_logo2)
    ax_logo2.set_title("Central 60 bp Enhancer Grammar (WT-Projected Activating Profile)", fontweight="bold")
    ax_logo2.set_ylabel("WT Importance")
    ax_logo2.set_xlabel("Base Position (bp)")

    ax_swap = fig.add_subplot(gs[:, 1])
else:
    fig = plt.figure(figsize=(15, 6))
    gs = fig.add_gridspec(2, 2, width_ratios=[2.5, 1])

    ax_logo1 = fig.add_subplot(gs[0, 0])
    sub_logo = df_logo.iloc[max(0, best_offset - 4) : min(len(df_logo), best_offset + motif_len + 4)].reset_index(drop=True)
    logomaker.Logo(sub_logo, ax=ax_logo1)
    ax_logo1.set_title(f"Causal Motif: '{causal_seq}' (WT-Projected ISM - Activating Bases Positive)", fontweight="bold")
    ax_logo1.set_ylabel("WT Importance")

    ax_logo2 = fig.add_subplot(gs[1, 0])
    logomaker.Logo(df_logo, ax=ax_logo2)
    ax_logo2.set_title("Central 60 bp Enhancer Grammar (WT-Projected Activating Profile)", fontweight="bold")
    ax_logo2.set_ylabel("WT Importance")
    ax_logo2.set_xlabel("Base Position (bp)")

    ax_swap = fig.add_subplot(gs[:, 1])

cats = ["In Silico Ablation"]
deltas = [swap_results.get("target_ablation_delta", 0.0)]
if swap_results.get('target_swap_delta') is not None:
    cats.append("Ortholog Swap")
    deltas.append(swap_results["target_swap_delta"])
colors = ["#d7191c" if d < 0 else "#2b83ba" for d in deltas]
ax_swap.bar(cats, deltas, color=colors, width=0.4, edgecolor='black')
ax_swap.axhline(0, color="gray", linestyle="--")
ax_swap.set_ylabel("Predicted Δ Pre-MZT Accessibility (y_abs)")
ax_swap.set_title("Causal Element Perturbation Impact", fontweight="bold")
ax_swap.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()
fig4_path = res_dir / "fig4_motif_attribution_logo.png"
plt.savefig(fig4_path, dpi=300)
plt.show()
print(f"[+] Figure 4 exported to {fig4_path}")

---
## Block 7: Multi-Task `NTv3` Fine-Tuning, Baseline Benchmarks, QuantSeq 3'-End Scoring & Distal Knockout

We integrate the multi-task genomic foundation model (**Nucleotide Transformer v3**):
1. **Reuse Block 1's Processed SLAM-seq BigWigs:** Directly connects to Zebrafish nascent RNA SLAM-seq BigWigs ($0.75\text{ hpf}$, $2.5\text{ hpf}$, $3.0–3.5\text{ hpf}$, $4.5\text{ hpf}$) alongside ATAC BigWigs ($256\text{-cell}$, $\text{dome}$).
2. **Publication Baselines & Frozen-Head Benchmark (`NTV3_RUN_BASELINES = True`):**
   - Compares (1) Raw pretrained `NTv3_tracks`, (2) Frozen trunk + linear head (`frozen_head`), and (3) LoRA / full fine-tuning.
   - Reports early-vs-late timing AUROC (excluding `miR-430`), zero-shot Medaka `st9` ATAC AUROC, and zero-shot target-species `strictly_zygotic_minor_wave` & `slam_confirmed_minor_wave` AUROC.
3. **QuantSeq 3'-End Scoring for Distal In Silico Knockout:**
   - Evaluates whether knockout of the causal Oct-Sox motif identified in Block 6 impairs predicted nascent transcription across `TARGET_GENES`, reporting **both 3' terminal 1,000 bp / 3' UTR accumulation** (primary QuantSeq metric) and **all-exons sum**.
   - Handles `NTv3_tracks`'s central $37.5\%$ output crop.
4. **Option 2 Target-Species RNA-seq Head Ablation Toggle (`ENABLE_OPTION2_TARGET_RNA_ABLATION`):** Evaluates out-of-fold performance side-by-side with Option 1.

In [ ]:
# Launch Multi-Task NTv3 Epigenomics/SLAM-seq Fine-Tuning Task
if COMPUTE_PROFILE == "hpc_1mb" or NUM_GPUS > 1:
    ntv3_cmd = [
        "torchrun",
        "--nproc_per_node", str(NUM_GPUS),
        "scripts/train_ntv3.py",
        "--work-dir", WORK_DIR,
        "--model-id", NTV3_MODEL_ID,
        "--compute-profile", COMPUTE_PROFILE,
        "--context-len", str(NTV3_CONTEXT_LEN),
        "--train-mode", NTV3_TRAIN_MODE,
        "--batch-size", str(NTV3_BATCH_SIZE),
        "--grad-accum", str(NTV3_GRAD_ACCUM),
        "--task-name", "train_ntv3_zf",
        "--epochs", "3"
    ]
else:
    ntv3_cmd = [
        "python", "-u", "scripts/train_ntv3.py",
        "--work-dir", WORK_DIR,
        "--model-id", NTV3_MODEL_ID,
        "--compute-profile", COMPUTE_PROFILE,
        "--context-len", str(NTV3_CONTEXT_LEN),
        "--train-mode", NTV3_TRAIN_MODE,
        "--batch-size", str(NTV3_BATCH_SIZE),
        "--grad-accum", str(NTV3_GRAD_ACCUM),
        "--task-name", "train_ntv3_zf",
        "--epochs", "3"
    ]
if NTV3_RUN_BASELINES:
    ntv3_cmd.append("--run-baselines")
if ENABLE_OPTION2_TARGET_RNA_ABLATION:
    ntv3_cmd.append("--enable-option2-target-rna")

env_vars = {}
if HF_TOKEN:
    env_vars["HF_TOKEN"] = HF_TOKEN

tm.submit("train_ntv3_zf", ntv3_cmd, env_vars=env_vars, depends_on=["prep_atac_danio_rerio", "prep_slamseq_zf"], force_rerun=FORCE_RERUN)
display(tm.status())

In [ ]:
# Real Benchmark Summary & QuantSeq 3'-End Distal In Silico Knockout (Zero-Fabrication Policy)
from scripts.infer_and_attribute import quantseq_3p_gene_scoring
from pyfaidx import Fasta
import matplotlib.pyplot as plt

bench_tsv = Path(WORK_DIR) / "models" / "ntv3" / "ntv3_baselines_benchmark.tsv"
if bench_tsv.exists():
    df_baselines = pd.read_csv(bench_tsv, sep="\t")
    print("=" * 80)
    print("[*] Publication Cross-Architecture & Baselines Benchmark:")
    print(df_baselines.to_string())
    print("=" * 80)
else:
    print("[*] NTv3 benchmark table not yet generated (requires completed model training).")
    print("    Zero-mock policy strictly enforced: skipping hardcoded benchmark table display.")

# Safe summary metric fallback loading from TSVs placed ABOVE knockout block so fresh kernel never raises NameError
if "metrics_strict" not in locals() or "metrics_slam" not in locals():
    scored_tsv = res_dir / "target_genes_scored_metrics.tsv"
    if scored_tsv.exists():
        df_sc = pd.read_csv(scored_tsv, sep="\t")
        from scripts.infer_and_attribute import benchmark_transfer_metrics
        metrics_strict = benchmark_transfer_metrics(df_sc, pos_category="strictly_zygotic_minor_wave", ctrl_category="control_non_activated")
        metrics_slam = benchmark_transfer_metrics(df_sc, pos_category="slam_confirmed_minor_wave", ctrl_category="control_non_activated")
    else:
        metrics_strict = {"roc_auc": 0.0}
        metrics_slam = {"roc_auc": 0.0, "n_pos": 0}

if "causal_seq" not in locals():
    enh_tsv = res_dir / f"{TARGET_LOCUS_NAME}_candidate_enhancers.tsv"
    causal_seq = "N/A"
    if enh_tsv.exists():
        df_e = pd.read_csv(enh_tsv, sep="\t")
        if not df_e.empty and "center" in df_e.columns:
            causal_seq = f"Summit_{int(df_e.iloc[0]['center'])}"

if "swap_results" not in locals():
    swap_tsv = res_dir / "evolutionary_swap_summary.tsv"
    if swap_tsv.exists():
        swap_results = pd.read_csv(swap_tsv, sep="\t").iloc[0].to_dict()
    else:
        swap_results = {"target_ablation_delta": 0.0}

if "df_counts" not in locals():
    gene_sets_tsv = res_dir / f"{TARGET_LOCUS_NAME}_gene_sets.tsv"
    if gene_sets_tsv.exists():
        df_counts = pd.read_csv(gene_sets_tsv, sep="\t")
    else:
        df_counts = pd.DataFrame([{"category": "unclassified"}])

# 2. QuantSeq 3'-End Distal In Silico Knockout on TARGET_GENES
ntv3_ckpt = Path(WORK_DIR) / "models" / "ntv3" / "ntv3_best_checkpoint.pt"
last_enc_wt = None
last_slam_idx = None
ntv3_model = None

if not ntv3_ckpt.exists():
    print(f"[*] NTv3 checkpoint {ntv3_ckpt} not trained yet. Skipping forward scoring until training completes.")
else:
    print(f"[+] Found real trained NTv3 checkpoint: {ntv3_ckpt}")
    target_genome = Fasta(TARGET_GENOME_FA, as_raw=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    autocast_dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32

    try:
        from scripts.train_ntv3 import build_ntv3_model
        with open(ntv3_ckpt.parent / "ntv3_train_config.json") as f:
            n_cfg = json.load(f)
        ntv3_model, ntv3_tok = build_ntv3_model(
            model_id=n_cfg.get("model_id", NTV3_MODEL_ID),
            num_tracks=len(n_cfg.get("tracks", [])),
            train_mode=n_cfg.get("train_mode", "lora"),
            context_len=NTV3_CONTEXT_LEN,
            grad_checkpointing=False,
            hf_token=HF_TOKEN
        )
        payload = torch.load(ntv3_ckpt, map_location=device, weights_only=True)
        ntv3_model.load_state_dict(payload["model_state"])
        ntv3_model.to(device)
        ntv3_model.eval()

        print("[*] Running QuantSeq 3'-end distal knockout and motif perturbation across TARGET_GENES...")
        for tg in TARGET_GENES:
            tg_sub = df_counts[df_counts["gene_name"].str.lower() == tg.lower()]
            if tg_sub.empty:
                continue
            r = tg_sub.iloc[0]
            g_chr, g_st, g_en, g_strand = str(r["chrom"]), int(r["start"]), int(r["end"]), str(r["strand"])
            g_len = g_en - g_st
            # Center window so BOTH the causal motif is within [w_st, w_en]
            # AND the gene's 3' TES is within NTv3's central 37.5% output crop
            tes = g_en if g_strand == "+" else g_st
            crop_len = int(NTV3_CONTEXT_LEN * 0.375)
            crop_start = int(NTV3_CONTEXT_LEN * 0.3125)
            crop_end = crop_start + crop_len

            # Exact genomic coordinates of the causal motif from Block 6
            motif_genomic_start = (best_pos - window_bp // 2) + ism_region_start + best_offset if "best_pos" in locals() and "ism_region_start" in locals() else (g_st + g_en) // 2
            motif_genomic_end = motif_genomic_start + (len(causal_seq) if "causal_seq" in locals() else 10)

            # Center window on TES crop center
            w_st = max(0, tes - (crop_start + crop_len // 2))
            w_en = w_st + NTV3_CONTEXT_LEN

            # Ensure motif is also inside [w_st, w_en]
            if motif_genomic_start < w_st:
                shift = w_st - motif_genomic_start + 100
                w_st = max(0, w_st - shift)
                w_en = w_st + NTV3_CONTEXT_LEN
            elif motif_genomic_end > w_en:
                shift = motif_genomic_end - w_en + 100
                w_st = w_st + shift
                w_en = w_st + NTV3_CONTEXT_LEN

            dist_tes_motif = abs(tes - motif_genomic_start)
            if dist_tes_motif > NTV3_CONTEXT_LEN:
                print(f"    [!] Distance between {tg} TES ({tes}) and motif ({motif_genomic_start}) is {dist_tes_motif:,} bp (exceeds {NTV3_CONTEXT_LEN:,} bp context)")
            else:
                print(f"    [+] {tg}: Context window {g_chr}:{w_st:,}-{w_en:,} spans both TES and distal causal motif (distance = {dist_tes_motif:,} bp)")

            crop_genomic_start = w_st + crop_start
            crop_genomic_end = w_st + crop_end

            is_tes_in_crop = (crop_genomic_start <= tes <= crop_genomic_end)
            is_motif_in_window = (w_st <= motif_genomic_start <= w_en)
            if not is_tes_in_crop:
                print(f"    [!] Warning: Gene {tg} 3' TES ({tes}) falls outside model central crop [{crop_genomic_start}, {crop_genomic_end}]. 3'-end delta may be attenuated.")
            if not is_motif_in_window:
                print(f"    [!] Warning: Causal motif ({motif_genomic_start}) falls outside model window [{w_st}, {w_en}].")

            # Preserve soft-masking in WT sequence and pad if shorter than NTV3_CONTEXT_LEN
            wt_seq = str(target_genome[g_chr][w_st:w_en])
            if len(wt_seq) < NTV3_CONTEXT_LEN:
                wt_seq += "N" * (NTV3_CONTEXT_LEN - len(wt_seq))

            enc_wt = ntv3_tok(wt_seq, add_special_tokens=False, return_tensors="pt", max_length=NTV3_CONTEXT_LEN, truncation=True, padding="max_length")
            last_enc_wt = enc_wt

            with torch.no_grad():
                with torch.amp.autocast(device_type=device, dtype=autocast_dtype):
                    preds_wt = ntv3_model(enc_wt["input_ids"].to(device)).float().squeeze(0).cpu().numpy()

            # Ablate top causal motif using exact genomic offset
            abl_seq = list(wt_seq)
            m_pos = motif_genomic_start - w_st
            m_len = len(causal_seq) if "causal_seq" in locals() else 10
            if 0 <= m_pos and m_pos + m_len <= len(abl_seq):
                abl_seq[m_pos:m_pos + m_len] = ["N"] * m_len
                enc_abl = ntv3_tok("".join(abl_seq), add_special_tokens=False, return_tensors="pt", max_length=NTV3_CONTEXT_LEN, truncation=True, padding="max_length")
                with torch.no_grad():
                    with torch.amp.autocast(device_type=device, dtype=autocast_dtype):
                        preds_abl = ntv3_model(enc_abl["input_ids"].to(device)).float().squeeze(0).cpu().numpy()

                tracks_list = n_cfg.get("tracks", [])
                target_track_name = "slam_2.5hpf_fwd" if g_strand == "+" else "slam_2.5hpf_rev"
                slam_track_idx = tracks_list.index(target_track_name) if target_track_name in tracks_list else 4
                last_slam_idx = slam_track_idx

                scores_wt = quantseq_3p_gene_scoring(
                    preds_wt[:, slam_track_idx],
                    gene_span_len=g_len,
                    strand=g_strand,
                    resolution_bp=1,
                    crop_genomic_start=crop_genomic_start,
                    crop_genomic_end=crop_genomic_end,
                    gene_start=g_st,
                    gene_end=g_en
                )
                scores_abl = quantseq_3p_gene_scoring(
                    preds_abl[:, slam_track_idx],
                    gene_span_len=g_len,
                    strand=g_strand,
                    resolution_bp=1,
                    crop_genomic_start=crop_genomic_start,
                    crop_genomic_end=crop_genomic_end,
                    gene_start=g_st,
                    gene_end=g_en
                )

                if swap_results.get("ortholog_aligned_motif"):
                    swap_seq = list(wt_seq)
                    o_motif = swap_results["ortholog_aligned_motif"]
                    if len(o_motif) == m_len:
                        swap_seq[m_pos:m_pos + m_len] = list(o_motif)
                        enc_swap = ntv3_tok("".join(swap_seq), add_special_tokens=False, return_tensors="pt", max_length=NTV3_CONTEXT_LEN, truncation=True, padding="max_length")
                        with torch.no_grad():
                            with torch.amp.autocast(device_type=device, dtype=autocast_dtype):
                                preds_swap = ntv3_model(enc_swap["input_ids"].to(device)).float().squeeze(0).cpu().numpy()
                        scores_swap = quantseq_3p_gene_scoring(
                            preds_swap[:, slam_track_idx],
                            gene_span_len=g_len,
                            strand=g_strand,
                            resolution_bp=1,
                            crop_genomic_start=crop_genomic_start,
                            crop_genomic_end=crop_genomic_end,
                            gene_start=g_st,
                            gene_end=g_en
                        )
                        print(f"    Gene {tg}: Ortholog Swap Nascent 3' UTR = {scores_swap['nascent_3p_1000bp']:.3f} (Δ = {scores_swap['nascent_3p_1000bp'] - scores_wt['nascent_3p_1000bp']:+.3f})")
                print(f"    Gene {tg}: WT Nascent 3' UTR = {scores_wt['nascent_3p_1000bp']:.3f} | Ablated = {scores_abl['nascent_3p_1000bp']:.3f} (Δ = {scores_abl['nascent_3p_1000bp'] - scores_wt['nascent_3p_1000bp']:+.3f})")

        # 3. NTv3 Input-Gradient Saliency via Forward Hook on Trunk Input Embeddings
        if last_enc_wt is not None and ntv3_model is not None:
            print(f"[*] Computing NTv3 input-gradient saliency over {TARGET_LOCUS_NAME} enhancer...")
            try:
                trunk = ntv3_model.trunk if hasattr(ntv3_model, "trunk") else ntv3_model
                embed_layer = None
                if hasattr(trunk, "get_input_embeddings"):
                    embed_layer = trunk.get_input_embeddings()
                elif hasattr(trunk, "embeddings") and hasattr(trunk.embeddings, "word_embeddings"):
                    embed_layer = trunk.embeddings.word_embeddings

                if embed_layer is not None:
                    captured_embs = []
                    def hook_fn(module, inp, out):
                        if isinstance(out, torch.Tensor):
                            out.retain_grad()
                            captured_embs.append(out)

                    hook_handle = embed_layer.register_forward_hook(hook_fn)
                    try:
                        ntv3_model.zero_grad()
                        inp_ids = last_enc_wt["input_ids"].to(device)
                        with torch.amp.autocast(device_type=device, dtype=autocast_dtype):
                            out_tracks = ntv3_model(input_ids=inp_ids)
                            loss_sal = out_tracks[:, :, 0].sum()
                            if last_slam_idx is not None and last_slam_idx < out_tracks.shape[-1]:
                                loss_sal = loss_sal + out_tracks[:, :, last_slam_idx].sum()
                        loss_sal.backward()
                        if captured_embs and captured_embs[0].grad is not None:
                            sal = captured_embs[0].grad.abs().sum(dim=-1).squeeze(0).cpu().numpy()
                            print(f"[+] NTv3 Saliency computed: max={float(sal.max()):.4f}, mean={float(sal.mean()):.4f}")
                    finally:
                        hook_handle.remove()
            except Exception as e_sal:
                print(f"[-] Notice computing NTv3 saliency: {e_sal}")

    except Exception as e:
        print(f"[-] Notice evaluating NTv3 checkpoint: {e}")

# Export Final Study Summary Metrics
summary_metrics_file = res_dir / f"{TARGET_LOCUS_NAME}_summary_metrics.tsv"
df_summary = pd.DataFrame([
    {"metric": "strictly_zygotic_minor_wave_genes", "value": int((df_counts["category"] == "strictly_zygotic_minor_wave").sum())},
    {"metric": "slam_confirmed_minor_wave_genes", "value": int(metrics_slam.get("n_pos", 0))},
    {"metric": "zero_shot_strictly_zygotic_auroc", "value": float(metrics_strict.get("roc_auc", 0.0))},
    {"metric": "zero_shot_slam_confirmed_auroc", "value": float(metrics_slam.get("roc_auc", 0.0))},
    {"metric": "top_causal_motif", "value": causal_seq},
    {"metric": "evolutionary_ablation_delta", "value": float(swap_results.get("target_ablation_delta", 0.0) if swap_results.get("target_ablation_delta") is not None else 0.0)}
])
df_summary.to_csv(summary_metrics_file, sep="\t", index=False)
print(f"[+] Final study summary metrics written to {summary_metrics_file}")